# Cardiac Involvement in Pediatric Acute Encephalopathy and Infection-Triggered Encephalopathy Syndromes

**Question.** In critically ill children with acute encephalopathy, how common are myocardial injury, myocardial
dysfunction, and repolarization abnormalities; do they differ across infection-triggered encephalopathy syndromes (ITES,
2024 international consensus); and does early myocardial injury identify children who will deteriorate, beyond the organ
support already in place when the troponin is reported?

**How to run.** Place this notebook in the same folder as the two research-platform exports
(the main export with sheet `病案首页基本信息`, and the supplementary export with sheet `病案首页其他信息`) and the
frozen dataset folder `AE_freeze_v1.2`, and run all cells. Outputs (tables, figures, cohort file) are written to `./output_aes_cardiac/`; the parsed exports are
cached in `./cache_aes_exports/` (shared with the ITES notebook).

**Data notes.**
* High-sensitivity cardiac troponin T (hs-cTnT; cobas e602, Roche Diagnostics; confirmed with the laboratory) is reported
  in ng/mL with a lower reporting limit of 0.003 ng/mL. The primary threshold is the assay's 99th-percentile upper
  reference limit (URL) of 0.014 ng/mL (`HSCTN_URL`, 14 ng/L); the laboratory report displays the reference upper limit
  rounded to 0.01 ng/mL. Twice the URL (28 ng/L) is analyzed as a sensitivity threshold.
* Critical discharge against medical advice, vasoactive infusion, invasive ventilation, and CPR orders follow the frozen dataset of this
  export (`AE_freeze_v1.2`, place its folder next to this notebook): the clinician-adjudicated critical-discharge label and the freeze's
  organ-support rules. The documentation rule for critical discharge is kept as a sensitivity analysis.
* Laboratory results stored only in the qualitative column (for example "<0.003" or ">9000") are read as the reportable
  limit and flagged as censored; results without either value are ignored.
* No laboratory sampling times were exported; report times are used as the timing proxy.
* Syndrome groups come from the ITES re-classification (same code as `AES_ITES_classification.ipynb`); adjudicated
  categories are used automatically when `output_ites/ites_adjudicated.csv` exists.
* Every number is computed from the exported data; nothing is imputed, simulated, or hard-coded.


In [ ]:
import re, json, warnings, collections
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
SEED = 20261009
rng = np.random.default_rng(SEED)
HSCTN_URL = 0.014       # ng/mL: 99th-percentile URL of the hs-cTnT assay (Roche cobas e602; report displays 0.01 after rounding)
HSCTN_URL_ALT = 2 * HSCTN_URL   # ng/mL: sensitivity threshold (twice the URL)
URL_NGL, ALT_NGL = int(round(HSCTN_URL * 1000)), int(round(HSCTN_URL_ALT * 1000))
from decimal import Decimal, ROUND_HALF_UP

def rnd(x, d=1):
    # Conventional half-up rounding for displayed numbers (avoids binary-float and half-to-even artifacts).
    return str(Decimal(f'{float(x):.10f}').quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP))

HERE = Path.cwd()
OUT = HERE / 'output_aes_cardiac'
CACHE = HERE / 'cache_aes_exports'
FIG = OUT / 'figures'
for p in (OUT, CACHE, FIG):
    p.mkdir(exist_ok=True)

def pick_font():
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']:
        if name in available:
            return name
    return 'sans-serif'

FONT = pick_font()
plt.rcParams.update({
    'font.family': FONT, 'font.size': 13, 'axes.labelsize': 14, 'axes.labelweight': 'bold',
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False, 'axes.linewidth': 1.0,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print('Figure font:', FONT, '(Arial is used whenever it is installed)')

In [ ]:
# Locate and load both exports (about 2.5 minutes on the first run; cached afterwards).
KEY_SHEET, SUPP_SHEET = '病案首页基本信息', '病案首页其他信息'

def find_workbooks(folder):
    main = supp = None
    for p in sorted(folder.glob('*.xlsx')):
        if p.name.startswith('~$'):
            continue
        try:
            wb = openpyxl.load_workbook(p, read_only=True); names = wb.sheetnames; wb.close()
        except Exception:
            continue
        if SUPP_SHEET in names:
            supp = p
        elif KEY_SHEET in names:
            main = p
    if main is None or supp is None:
        raise FileNotFoundError('Both the main export and the supplementary export must be in this folder.')
    return main, supp

def load_book(path, cache_name):
    f = CACHE / cache_name
    if f.exists() and f.stat().st_mtime > path.stat().st_mtime:
        return pd.read_pickle(f)
    book = pd.read_excel(path, sheet_name=None, dtype=object, engine='openpyxl')
    pd.to_pickle(book, f)
    return book

WB, WB_SUPP = find_workbooks(HERE)
sheets = load_book(WB, 'main.pkl')
supp = load_book(WB_SUPP, 'supplement.pkl')
print('Main export:', WB.name, '| sheets:', len(sheets))
print('Supplementary export:', WB_SUPP.name, '| sheets:', len(supp))

In [ ]:
# Admissions and free-text discharge diagnoses (same parser as the data-asset notebook).
def split_top(s):
    out, buf, depth = [], '', 0
    for ch in s:
        if ch in '(（[［':
            depth += 1
        elif ch in ')）]］':
            depth = max(0, depth - 1)
        if ch in ',，;；、' and depth == 0:
            out.append(buf); buf = ''
        else:
            buf += ch
    out.append(buf)
    return out

def parse_dx(s, kind='出院'):
    if not isinstance(s, str):
        return []
    s = re.sub(kind + r'诊断\s*\d*\s*[:：]', '\n', s.replace('\r', '\n'))
    items = []
    for line in s.split('\n'):
        line = line.strip()
        if not line:
            continue
        for part in re.split(r'(?:(?<=^)|(?<=[^\d\.]))\d{1,2}\s*[\.．、]\s*(?=\D)', line):
            for q in split_top(part):
                q = q.strip(' 　\t。.')
                if q:
                    items.append(q)
    return items

def uniq(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

fp = sheets['病案首页基本信息'].copy()
fp['vis'] = fp['科研就诊编号'].astype(str)
fp['pid'] = fp['科研患者编号'].astype(str)
fp['admit'] = pd.to_datetime(fp['入院日期'], errors='coerce')
fp['discharge'] = pd.to_datetime(fp['出院日期'], errors='coerce')
fp['age_y'] = pd.to_numeric(fp['年龄（岁）'], errors='coerce')

dis = sheets['住院病历出院记录'].copy()
dis['dx_list'] = dis['出院诊断'].apply(parse_dx)
longest = lambda s: max([x for x in s if isinstance(x, str)] or [''], key=len)
agg = dis.groupby('科研就诊编号').agg(dx_list=('dx_list', lambda s: uniq(sum(s.tolist(), []))),
                                    course=('诊疗经过', longest), dstatus=('出院情况', longest)).reset_index()
agg['vis'] = agg['科研就诊编号'].astype(str)
adm = fp.merge(agg.drop(columns=['科研就诊编号']), on='vis', how='left')
adm['has_disrec'] = adm['dx_list'].apply(lambda x: isinstance(x, list))
adm['dx_list'] = adm['dx_list'].apply(lambda x: x if isinstance(x, list) else [])
for c in ['course', 'dstatus']:
    adm[c] = adm[c].fillna('')
print('Admissions:', len(adm), '| patients:', adm.pid.nunique())

In [ ]:
# Provisional syndrome labels (rule-based; identical to the data-asset notebook).
ORDER = [
    ('Metabolic_Leigh', r'亚急性坏死性|Leigh'),
    ('ANE', r'坏死性\)?脑|脑病[^,，;；]{0,14}坏死性|急性坏死性(?!.{0,4}(?:肠|筋膜|牙龈|肺|淋巴|胰))'),
    ('HSES_ASEM', r'出血性休克脑病|休克脑病|HSES'),
    ('Reye', r'瑞氏|Reye|瑞夷'),
    ('AESD', r'AESD|双相发作|双向发作|迟发性弥散|后期弥散|后期弥漫|晚期弥散|迟发性扩散'),
    ('FIRES', r'FIRES|FIERS|热性感染相关|发热感染相关|热性敏感性|感染相关性癫痫综合征|AERRPS|难治性反复部分性'),
    ('MERS', r'胼胝体压部|MERS|RESLES|胼胝体压迫|可逆性胼胝体|胼胝体可逆'),
    ('Septic_encephalopathy', r'脓毒症性\)?脑病|脓毒性\)?脑病|脓毒症相关性脑病|脓毒症脑病'),
    ('Pertussis_encephalopathy', r'百日咳脑病'),
    ('Influenza_encephalopathy', r'流感相关性\)?脑病|流感脑病|流感相关脑病|流感相关性脑炎'),
    ('Toxic_encephalopathy', r'中毒性脑病'),
    ('Bilirubin', r'胆红素脑病|核黄疸'),
    ('Neonatal_HIE_NE', r'新生儿.*脑病|新生儿缺氧缺血|新生儿缺血缺氧|\(新生儿\)缺氧'),
    ('HIE_other', r'缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤'),
    ('DEE', r'癫痫性脑病|ESES|CSWS|睡眠中癫痫性电持续|Lennox|大田原|暴发抑制|发育性.*脑病|癫痫性\)脑病|West综合征|婴儿痉挛'),
    ('Metabolic', r'代谢性脑病|甘氨酸脑病|低血糖性脑病|低血糖昏迷|肝性脑病|尿毒症性脑病|Wernicke|韦尼克|线粒体脑|高氨血症脑病'),
    ('PRES_hypertensive', r'可逆性后部|PRES|高血压脑病|后部可逆'),
    ('Lupus', r'狼疮性脑病|狼疮脑病|神经精神狼疮|狼疮脑'),
    ('Autoimmune', r'自身免疫性脑炎|自体免疫性脑炎|NMDA|GABA|LGI1|免疫介导性脑炎|自身免疫相关性脑病|桥本脑病|自身免疫性\)?脑病|免疫性？\)|ADEM|急性播散性脑脊髓炎'),
    ('Infectious_encephalitis', r'病毒性脑炎|脑膜脑炎|^\(?病毒性\)?脑炎|化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|^脑炎|汉坦病毒脑炎|疱疹病毒性脑炎|乙型脑炎'),
    ('Leukoencephalopathy', r'白质脑病|脑白质病'),
    ('AES_unspecified', r'急性脑病|急性\)\s*脑病|^脑病|弥散性脑病|中枢神经系统感染相关性脑病'),
]
AES = ['ANE', 'HSES_ASEM', 'Reye', 'AESD', 'FIRES', 'MERS', 'Septic_encephalopathy', 'Pertussis_encephalopathy',
       'Influenza_encephalopathy', 'Toxic_encephalopathy', 'AES_unspecified']
SEQ = re.compile(r'后遗症|恢复期|随诊|复查|治疗后|个人史')

def label_string(x):
    if '胼胝体' in x and re.search(r'先天性|畸形|发育不良|缺如', x) and '脑病' not in x:
        return None
    for k, pat in ORDER:
        if re.search(pat, x):
            return k
    return None

def visit_labels(L):
    out = {}
    for x in L:
        k = label_string(x)
        if k:
            out.setdefault(k, []).append(x)
    return out

def aes_info(lab):
    keys = [k for k in lab if k in AES]
    if not keys:
        return pd.Series([False, None, None])
    strs = sum([lab[k] for k in keys], [])
    acute = any(not SEQ.search(s) for s in strs)
    return pd.Series([True, next(k for k in AES if k in keys), acute])

# Coded discharge diagnoses are used only where no discharge summary exists (mostly in-hospital deaths).
dxc = supp['病案出院诊断归一'].copy(); dxc['vis'] = dxc['科研就诊编号'].astype(str)
coded = dxc.groupby('vis')['诊断疾病名称'].apply(lambda s: [str(x) for x in s])
adm['coded_list'] = adm.vis.map(coded).apply(lambda x: x if isinstance(x, list) else [])
adm['lab'] = [visit_labels(L if hd else C) for L, C, hd in zip(adm['dx_list'], adm['coded_list'], adm['has_disrec'])]
adm[['aes', 'aes_sub', 'aes_acute']] = adm['lab'].apply(aes_info)
adm['all_dx_text'] = [' | '.join(L + C) for L, C in zip(adm['dx_list'], adm['coded_list'])]
print('AES-labeled admissions:', int(adm.aes.sum()))

In [ ]:
# Index admission: the first acute-phase AES admission per patient; dispositions from the supplement.
oth = supp['病案首页其他信息'].copy(); oth['vis'] = oth['科研就诊编号'].astype(str)
PICU_WARD = re.compile(r'重症医学|儿童重症监护')     # PICU wards (surgical ICU not included)
trf = supp['患者转科信息'].copy(); trf['vis'] = trf['科研就诊编号'].astype(str)
trf['t'] = pd.to_datetime(trf['转科时间'], errors='coerce')
trf = trf.merge(adm[['vis', 'admit', 'discharge']], on='vis', how='inner')
trf = trf[(trf.t >= trf.admit - pd.Timedelta(days=1)) & (trf.t <= trf.discharge + pd.Timedelta(days=1))]
PICU_TRANSFER = set(trf[trf['转出科室'].astype(str).str.contains(PICU_WARD) | trf['转入科室'].astype(str).str.contains(PICU_WARD)].vis)
vinfo = supp['患者就诊信息'].copy(); vinfo['vis'] = vinfo['科研就诊编号'].astype(str)
adm = adm.merge(oth[['vis', '离院方式']], on='vis', how='left').merge(vinfo[['vis', '转归情况']], on='vis', how='left')
adm['covered'] = adm.vis.isin(set(oth.vis))
adm['death'] = adm['离院方式'].eq('死亡') | adm['转归情况'].eq('死亡')
adm['ama'] = adm['离院方式'].eq('非医嘱离院')

# Critical-condition wording at the end of the discharge summary (supplement version, else main export).
WD = re.compile(r'放弃治疗|放弃抢救|自动出院|拒绝.{0,6}(?:气管插管|治疗)|随时.{0,6}死亡|死亡风险|心跳骤停.{0,10}(?:风险|可能)|极可能.{0,4}死亡|带气管插管出院|拔除气管插管|撤机')
CRIT = re.compile(r'危重|病重|预后差|预后极差|预后不良|昏迷|电静息|脑死亡|脑疝')
ds = supp['住院病历出院记录'].copy(); ds['vis'] = ds['科研就诊编号'].astype(str)
ds_txt = ds.groupby('vis').apply(lambda g: ' '.join(str(x) for x in list(g['诊疗经过'])[-1:] + list(g['出院情况'])))
def critical_wording(row):
    t = ds_txt.get(row.vis, None)
    t = str(t) if t is not None else (row.course + ' ' + row.dstatus)
    tail = t[-600:]
    return bool(WD.search(tail) and CRIT.search(tail))

adm = adm.sort_values(['pid', 'admit'])
rows = []
for pid, g in adm[adm.aes == True].groupby('pid'):
    sub = next(k for k in AES if k in set(g.aes_sub))
    ac = g[g.aes_acute == True]
    r = ac.iloc[0] if len(ac) else g.iloc[0]
    wards = ' '.join(str(x) for x in [r['入院科别'], r['出院科别'], r['转科科别']] if isinstance(x, str))
    rows.append({'patient': pid, 'vis': r.vis, 'subtype': sub, 'acute_index': bool(r.aes_acute), 'admit': r.admit,
                 'discharge': r.discharge, 'year': r.admit.year, 'age_y': r.age_y, 'male': r['性别'] == '男性',
                 'icu': bool(PICU_WARD.search(wards)) or r.vis in PICU_TRANSFER, 'covered': bool(r.covered), 'death': bool(r.death), 'ama': bool(r.ama),
                 'disposition': r['离院方式'], 'has_disrec': bool(r.has_disrec), 'dx_text': r.all_dx_text,
                 'course': r.course, 'dstatus': r.dstatus})
coh = pd.DataFrame(rows)
idx = coh[coh.acute_index & coh.covered].copy().reset_index(drop=True)
idx['critical_ama_rule'] = idx.ama & idx.apply(critical_wording, axis=1)   # documentation rule (sensitivity label)

# Reference labels from the frozen dataset of this export (AE_freeze_v1.2): critical discharge against medical advice as adjudicated by a
# pediatric clinician (dama_critical_final; the freeze rule label is kept where the reviewer was unsure).
FREEZE_DIR = HERE / 'AE_freeze_v1.2'
if not (FREEZE_DIR / 'tables' / 'admission_summary.csv').exists():
    raise FileNotFoundError('Place the frozen dataset folder AE_freeze_v1.2 (with its tables/ subfolder) next to this notebook.')
FZ = pd.read_csv(FREEZE_DIR / 'tables' / 'admission_summary.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False).set_index('visit_id')
FZ_EVENTS = pd.read_csv(FREEZE_DIR / 'tables' / 'support_events.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False)
not_in_freeze = set(idx.vis) - set(FZ.index)
if not_in_freeze:
    raise ValueError(f'{len(not_in_freeze)} index admissions are missing from the frozen dataset.')

def fz_flag(col):
    return idx.vis.map(FZ[col].astype(str).str.lower().eq('true')).astype(bool)

idx['critical_ama'] = idx.ama & fz_flag('dama_critical_final')
idx['poor'] = idx.death | idx.critical_ama
print('AES patients:', len(coh), '| acute index admissions:', int(coh.acute_index.sum()),
      '| with a recorded disposition:', len(idx))
print('Deaths:', int(idx.death.sum()), '| critical AMA (clinician-adjudicated):', int(idx.critical_ama.sum()),
      '| by the documentation rule:', int(idx.critical_ama_rule.sum()), '| death or critical AMA:', int(idx.poor.sum()))

In [ ]:
# Laboratory, ECG, and echocardiography measurements (both exports, de-duplicated by report number).
def parse_lab_value(quant, qual=None):
    # Numeric result; when only the qualitative column holds it (for example "<0.003" or ">9000"), the value or the
    # reportable limit is used and the result is flagged as censored.
    v = pd.to_numeric(quant, errors='coerce')
    if qual is None:
        return v, pd.Series(False, index=v.index)
    q = qual.astype(str).str.strip().str.replace('〈', '<').str.replace('〉', '>').str.replace('＜', '<').str.replace('＞', '>')
    m = q.str.extract(r'^([<>]=?)?\s*(\d*\.?\d+)$')
    qv = pd.to_numeric(m[1], errors='coerce')
    use = v.isna() & qv.notna()
    return v.where(~use, qv), use & m[0].notna()

def qual_col(df, col):
    base = col.replace('-定量结果', '')
    for c in (base + '-定性结果', base + '-定性结果显示'):
        if c in df.columns:
            return df[c]
    return None

def lab_long(sheet, col):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col in book[sheet].columns:
            df = book[sheet]
            val, cens = parse_lab_value(df[col], qual_col(df, col))
            parts.append(pd.DataFrame({'report': df['报告编号'].astype(str), 'vis': df['就诊编号'].astype(str),
                                       'value': val, 'censored': cens,
                                       'time': pd.to_datetime(df['报告时间'], errors='coerce')}))
    x = pd.concat(parts).dropna(subset=['value', 'time']).drop_duplicates('report')
    return x[x.vis.isin(set(idx.vis))]

def summarize(long, hours, prefix):
    x = long.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
    within = x[(x.h >= -12) & (x.h <= hours)].sort_values('time')
    first = within.groupby('vis').first()
    out = pd.DataFrame({f'{prefix}_early': first.value, f'{prefix}_early_time': first.time})
    out[f'{prefix}_peak'] = x.groupby('vis').value.max()
    out[f'{prefix}_first_time'] = x.sort_values('time').groupby('vis').time.first()
    return out

tnt = summarize(lab_long('实验室检查_血液心肌酶谱测定', '超敏心肌钙蛋白定量-定量结果'), 48, 'tnt')
ckmb = summarize(lab_long('实验室检查_血液心肌酶谱测定', '肌酸激酶同工酶-ＭＢ定量-定量结果'), 48, 'ckmb')
bnp = summarize(lab_long('实验室检查_血液N末端B型钠尿肽前体测定', 'N末端B型利钠肽原定量-定量结果'), 48, 'bnp')
lac = summarize(lab_long('实验室检查_血液乳酸测定', '乳酸定量-定量结果'), 48, 'lac')

ecg = supp['心电图报告'].copy(); ecg['vis'] = ecg['科研就诊编号'].astype(str)
ecg['time'] = pd.to_datetime(ecg['心电检查时间'], errors='coerce')
ecg['HR'] = pd.to_numeric(ecg['HR'], errors='coerce'); ecg['QT'] = pd.to_numeric(ecg['QT'], errors='coerce')
ecg = ecg[ecg.vis.isin(set(idx.vis)) & ecg.HR.between(30, 300) & ecg.QT.between(150, 700)].copy()
ecg['value'] = ecg.QT / np.cbrt(60 / ecg.HR)                      # Fridericia
ecg['QTcB'] = ecg.QT / np.sqrt(60 / ecg.HR)                        # Bazett (equals the machine QTc)
ecg['report'] = ecg.index.astype(str)
qtc = summarize(ecg[['report', 'vis', 'value', 'time']], 48, 'qtcf')
qtc['qtcb_peak'] = ecg.groupby('vis').QTcB.max()
qtc['qtcb_early'] = summarize(ecg[['report', 'vis', 'QTcB', 'time']].rename(columns={'QTcB': 'value'}), 48, 'qtcb')['qtcb_early']

echo = supp['儿童心脏彩超'].copy(); echo['vis'] = echo['科研就诊编号'].astype(str)
echo['time'] = pd.to_datetime(echo['检查时间'], errors='coerce'); echo['value'] = pd.to_numeric(echo['射血分数'], errors='coerce')
echo = echo[echo.vis.isin(set(idx.vis))].dropna(subset=['value', 'time'])
echo['report'] = echo.index.astype(str)
x = echo.merge(idx[['vis', 'admit']], on='vis'); x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
ef = pd.DataFrame({'lvef_min': x.groupby('vis').value.min(),
                   'lvef_early_min': x[(x.h >= -24) & (x.h <= 72)].groupby('vis').value.min(),
                   'lvef_first_time': x.sort_values('time').groupby('vis').time.first()})

for t in (tnt, ckmb, bnp, lac, qtc, ef):
    idx = idx.merge(t, left_on='vis', right_index=True, how='left')
idx = idx.reset_index(drop=True)
idx['tnt_early_ngL'] = idx.tnt_early * 1000
idx['tnt_peak_ngL'] = idx.tnt_peak * 1000
print('Index admissions with early hs-cTn:', int(idx.tnt_early.notna().sum()), '| NT-proBNP:', int(idx.bnp_early.notna().sum()),
      '| ECG QT/HR:', int(idx.qtcf_early.notna().sum()), '| LVEF:', int(idx.lvef_min.notna().sum()))

In [ ]:
# Organ-support and severity variables, cardiac arrest timing, era.
proc = supp['病案手术操作归一'].copy(); proc['vis'] = proc['科研就诊编号'].astype(str)
proc['date'] = pd.to_datetime(proc['手术操作日期'], errors='coerce')
proc = proc[proc.vis.isin(set(idx.vis))]
code_ = proc['手术操作编码'].astype(str)
vent_proc = set(proc[code_.str.match(r'^96\.7|^96\.04')].vis)
cpr_proc = proc[code_.str.match(r'^99\.6')].groupby('vis').date.min()

orders = supp['住院非用药医嘱'].copy(); orders['vis'] = orders['科研就诊编号'].astype(str)
orders = orders[orders.vis.isin(set(idx.vis))]
vent_order = set(orders[orders['住院非用药医嘱名称'].astype(str).str.contains('气管插管')].vis)

meds = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                  supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds['vis'] = meds['科研就诊编号'].astype(str)
meds = meds[meds.vis.isin(set(idx.vis))]
name = meds['住院用药医嘱药品名称'].astype(str); route = meds['住院用药医嘱用法'].astype(str)
VASO = r'(?<!异丙)肾上腺素|多巴胺|多巴酚丁胺|米力农|垂体后叶|血管加压素'   # isoproterenol (异丙肾上腺素) excluded
BOLUS_OR_LOCAL = r'雾化|气管|静脉注射|静推|皮下|外用|局部|口服|滴眼|滴鼻'
vaso = set(meds[name.str.contains(VASO) & ~route.str.contains(BOLUS_OR_LOCAL)].vis)
DRUG = {'methylprednisolone': r'甲泼尼龙|甲强龙|甲基强的松龙', 'ivig': r'丙种球蛋白|免疫球蛋白', 'tocilizumab': r'托珠单抗'}
for k, p in DRUG.items():
    idx[k] = idx.vis.isin(set(meds[name.str.contains(p)].vis))

idx['vasoactive'] = idx.vis.isin(vaso)

# Time-resolved organ support (order timestamps) for the time-anchored analyses.
meds_t = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                    supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds_t['vis'] = meds_t['科研就诊编号'].astype(str)
meds_t = meds_t[meds_t.vis.isin(set(idx.vis))]
meds_t['t'] = pd.to_datetime(meds_t['住院用药医嘱开立时间'], errors='coerce')
nm, rt = meds_t['住院用药医嘱药品名称'].astype(str), meds_t['住院用药医嘱用法'].astype(str)
infusion = meds_t[nm.str.contains(VASO) & ~rt.str.contains(BOLUS_OR_LOCAL)]
pressor = infusion[~infusion['住院用药医嘱药品名称'].astype(str).str.contains('多巴酚丁胺|米力农')]
idx['vaso_time'] = idx.vis.map(infusion.groupby('vis').t.min())
idx['pressor_time'] = idx.vis.map(pressor.groupby('vis').t.min())
orders['t'] = pd.to_datetime(orders['住院非用药医嘱开始时间'], errors='coerce')
onames = orders['住院非用药医嘱名称'].astype(str)
inv = orders[onames.str.contains('气管插管|呼吸机辅助呼吸|机械通气') & ~onames.str.contains('无创|鼻塞|CPAP|高流量')]
vent_proc_date = proc[code_.str.match(r'^96\.7|^96\.04')].groupby('vis').date.min()
idx['vent_time'] = pd.concat([idx.vis.map(inv.groupby('vis').t.min()), idx.vis.map(vent_proc_date)], axis=1).min(axis=1)
idx['ventilation'] = idx.vis.isin(vent_proc | vent_order | set(inv.vis))
# Reference definitions: vasoactive infusion and invasive ventilation (presence and first time) follow the frozen dataset rules (AE_freeze_v1.2,
# definitions section 3: continuous-infusion routes only, pituitrin reported separately; planned-procedure intubation and ventilator orders with
# non-invasive ventilation evidence not counted). Vasopressor-only start excludes dobutamine and milrinone.
own = idx[['vasoactive', 'ventilation']].copy()
idx['vasoactive'] = fz_flag('vasoactive_ever')
idx['vaso_time'] = pd.to_datetime(idx.vis.map(FZ.vasoactive_first_time), errors='coerce')
va_ev = FZ_EVENTS[FZ_EVENTS.support.eq('vasoactive_infusion')].copy()
va_ev['t'] = pd.to_datetime(va_ev.event_time, errors='coerce')
idx['pressor_time'] = idx.vis.map(va_ev[~va_ev.detail.astype(str).isin(['dobutamine', 'milrinone'])].groupby('visit_id').t.min())
idx['ventilation'] = fz_flag('imv_ever')
idx['vent_time'] = pd.to_datetime(idx.vis.map(FZ.imv_first_time), errors='coerce')
print('Frozen-dataset rules vs order-text rules: vasoactive', int(idx.vasoactive.sum()), 'vs', int(own.vasoactive.sum()),
      '| invasive ventilation', int(idx.ventilation.sum()), 'vs', int(own.ventilation.sum()))
txt = idx.dx_text
idx['status_epilepticus'] = txt.str.contains('癫痫持续状态|惊厥持续状态')
idx['septic_shock'] = txt.str.contains('脓毒性休克|感染性休克')
idx['shock_dx'] = txt.str.contains('休克')
idx['lactate_ge4'] = idx.lac_early >= 4
idx['age_lt1'] = idx.age_y < 1

# Cardiac arrest / CPR: dated procedures, undated diagnoses, or narrative mention.
idx['cpr_date'] = idx.vis.map(cpr_proc)
ARREST = re.compile(r'心肺复苏|心脏骤停|心跳骤停|呼吸心跳骤停|心搏骤停|心脏停搏')
HEDGE = re.compile(r'可能|随时|风险|警惕|预防|避免|防止|告知|交代|谈话|签字|若|如出现|一旦|拒绝|放弃|不予')

def arrest_documented(text):
    # Affirmative narrative mention of cardiac arrest or CPR; clauses with warnings, counseling, or refusal are ignored.
    for sent in re.split(r'[。；;\n！!？?]', str(text)):
        for c in re.split(r'[，,]', sent):
            m = ARREST.search(c)
            if m and not HEDGE.search(c) and not re.search(r'无|未|否认', c[max(0, m.start() - 4):m.start()]):
                return True
    return False

idx['cpr_any'] = idx.cpr_date.notna() | txt.str.contains('心脏停搏|呼吸心跳骤停|心跳骤停|心搏骤停') | idx.course.apply(arrest_documented)
# CPR orders and procedure codes from the frozen dataset are added; the earliest dated evidence is the CPR date.
idx['cpr_any'] = idx.cpr_any | fz_flag('cpr_ever')
idx['cpr_date'] = pd.concat([idx.cpr_date, pd.to_datetime(idx.vis.map(FZ.cpr_first_time), errors='coerce').dt.normalize()], axis=1).min(axis=1)
def arrest_before(first_time):
    # True when an arrest/CPR is documented and cannot be shown to occur on a later calendar day than first_time.
    later = idx.cpr_date.notna() & first_time.notna() & (first_time.dt.normalize() < idx.cpr_date)
    return idx.cpr_any & ~later
idx['arrest_before_tnt'] = arrest_before(idx.tnt_early_time)
idx['arrest_before_echo'] = arrest_before(idx.lvef_first_time)
idx['arrest_before_ecg'] = arrest_before(idx.qtcf_early_time)
idx['arrest_before_bnp'] = arrest_before(idx.bnp_early_time)
idx['arrest_before_tnt_first'] = arrest_before(idx.tnt_first_time)

idx['era'] = pd.cut(idx.admit, [pd.Timestamp('2000-01-01'), pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-01'),
                                pd.Timestamp('2023-02-01'), pd.Timestamp('2100-01-01')],
                    labels=['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], right=False)
print(idx[['ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock', 'cpr_any', 'methylprednisolone', 'ivig', 'tocilizumab']].sum())

## ITES re-classification (2024 international consensus)

Each index admission is re-classified against the 2024 international consensus definitions of
infection-triggered encephalopathy syndromes (ITES; Sakuma et al., Dev Med Child Neurol 2024).
The algorithm only *proposes* a category from the exported evidence; every proposal is written to an
adjudication worksheet so that a pediatric neurologist can confirm or override it. If an adjudicated file
(`output_ites/ites_adjudicated.csv`, columns `vis` and `adjudicated_class`) is present, adjudicated
categories replace the algorithmic ones in all downstream analyses.

In [ ]:
# ITES evidence 1/4: neuroimaging features from MRI and head CT reports (clause-level negation, side-aware thalamic rule).
SENT = re.compile(r'[。；;\n！!？?]')
CLAUSE = re.compile(r'[，,]')
NEG_IMG = re.compile(r'未见|未发现|无明显|无异常|无确切|未显示|不明显|消失|吸收|未见确切|(?<!异常)正常|排除')
ABN = re.compile(r'异常信号|信号异常|异常密度|长T1|长T2|短T1|短T2|高信号|信号增高|信号灶|信号影|弥散受限|扩散受限|肿胀|肿大|水肿|坏死|出血'
                 r'|异常灶|病灶|低密度|密度减低|密度降低|异常改变')
DWI_POS = re.compile(r'弥散受限|扩散受限|弥散[^，,。；]{0,4}受限|DWI[^，,。；]{0,14}(?:高信号|信号增高|受限)|弥散加权[^，,。；]{0,8}高信号')
SIDE = re.compile(r'双侧|两侧|对称|左|右')
DIFFUSE_EDEMA = re.compile(   # global swelling, herniation or cisternal effacement
    r'全脑(?:肿胀|水肿)|弥漫性脑(?:肿胀|水肿)|广泛(?:性)?脑(?:肿胀|水肿)|脑组织弥漫性?肿胀|颅内广泛脑肿胀|全脑弥漫性肿胀'
    r'|大脑[、及和]?小脑半球[^。；]{0,24}肿胀|扁桃体[^，,。；]{0,6}(?:下疝|疝入)|脑疝形成|全脑.{0,6}脑疝'
    r'|脑池[^，,。；]{0,6}(?:闭塞|消失)|基底池[^，,。；]{0,6}(?:受压|闭塞|消失)'
    r'|脑沟[^，,。；]{0,4}(?:普遍|广泛|弥漫)[^，,。；]{0,4}(?:变浅|消失)|(?:普遍|广泛|弥漫)[^，,。；]{0,6}脑沟[^，,。；]{0,6}(?:变浅|消失)')
GWM_LOSS = re.compile(r'(?:双侧|两侧)大脑(?:半球)?[^。；]{0,16}灰白质(?:分界|界限)[^，,。；]{0,3}(?:不清|模糊|欠清)')
EFFACE = re.compile(r'脑沟[^，,。；]{0,6}(?:变浅|消失|受压)|脑池[^，,。；]{0,6}(?:受压|变窄|闭塞|消失)|脑室[^，,。；]{0,8}(?:变窄|变小|稍小|受压|缩小|显示不清)'
                    r'|外侧裂[^，,。；]{0,4}(?:闭塞|变窄)')
SUBCORT = re.compile(r'皮层下|皮质下|U形纤维|U纤维|皮髓质交界|灰白质交界')
LOBAR_WM = re.compile(r'(?:额|顶|颞|枕)叶?[^，,。；]{0,6}白质|大脑半球[^，,。；]{0,4}白质')
EXTENSIVE = re.compile(r'双侧|两侧|多发|弥漫|广泛|大脑半球|额顶|顶枕|额颞|颞顶|颞枕')
UNSURE = re.compile(r'似|可疑|[？?]')
HEMI = re.compile(r'(?<!双侧)(?<!两侧)(?:左|右)(?:侧)?大脑半球')
CHRONIC = re.compile(r'萎缩|软化|囊变|胶质增生|陈旧')


def clauses(text):
    for sent in SENT.split(str(text)):
        if sent.strip():
            yield sent, [c for c in CLAUSE.split(sent) if c.strip()]


def positive_clauses(text, region):
    # Clauses naming the region together with an abnormal descriptor and no negation.
    return [(sent, c) for sent, cls in clauses(text) for c in cls if re.search(region, c) and ABN.search(c) and not NEG_IMG.search(c)]


def thalamic_side(clause):
    # Nearest side marker before "丘脑" in its list item; otherwise inherited from the nearest preceding list item.
    items = clause.split('、')
    for i, it in enumerate(items):
        p = it.find('丘脑')
        if p < 0:
            continue
        marks = SIDE.findall(it[:p])
        if not marks:
            if '对称' in it[p:]:
                return 'bilateral'
            for prev in reversed(items[:i]):
                mm = SIDE.findall(prev)
                if mm:
                    marks = [mm[0]]
                    break
        if not marks:
            return 'unspecified'
        return 'unilateral' if marks[-1] in ('左', '右') else 'bilateral'
    return None


def find_thalamic(text, bilateral):
    for sent, c in positive_clauses(text, r'丘脑'):
        side = thalamic_side(c)
        if side is None:
            continue
        if side == 'unspecified' and re.search(r'对称|双侧丘脑|两侧丘脑', sent):
            side = 'bilateral'
        if not bilateral or side == 'bilateral':
            return c.strip()
    return ''


def find_splenial(text):
    for sent, cls in clauses(text):
        if '胼胝体' in sent and '压部' in sent:
            for c in cls:
                if '压部' in c and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c):
                    return sent.strip()
    m = re.search(r'[^。；]*(?:可逆性胼胝体压部|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病|病变))[^。；]*', str(text))
    if m and not re.search(r'未见|消失|吸收', m.group(0)):
        return m.group(0).strip()
    return ''


def find_subcortical_dwi(text, region=SUBCORT):
    # Extensive (bilateral or multilobar) white-matter lesions with restricted diffusion in the same sentence (bright-tree-type pattern).
    for sent, cls in clauses(text):
        if not region.search(sent) or CHRONIC.search(sent) or UNSURE.search(sent) or not EXTENSIVE.search(sent):
            continue
        reg = any(region.search(c) and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c) for c in cls)
        dwi = any(DWI_POS.search(c) and not NEG_IMG.search(c) for c in cls)
        if reg and dwi:
            return sent.strip()
    return ''


def find_diffuse_edema(text):
    # Global swelling, herniation or cisternal effacement; bilateral loss of gray-white differentiation counts only with effacement.
    efface = any(EFFACE.search(c) and not re.search(r'未见|无明显|无|未', c) for _, cls in clauses(text) for c in cls)
    for sent, cls in clauses(text):
        for c in cls:
            m = DIFFUSE_EDEMA.search(c)
            if m and not re.search(r'未见|无明显|无|未|排除', c[max(0, m.start() - 8):m.start()]):
                return sent.strip()
        m = GWM_LOSS.search(sent)
        if m and efface and not re.search(r'未见|无明显|无|未', sent[max(0, m.start() - 8):m.start()]):
            return sent.strip()
    return ''


def find_hemispheric(text):
    for sent, cls in clauses(text):
        if re.search(r'双侧大脑半球|两侧大脑半球', sent) or CHRONIC.search(sent):
            continue
        if HEMI.search(sent) and (re.search(r'肿胀|水肿', sent) or DWI_POS.search(sent)) and not NEG_IMG.search(sent):
            return sent.strip()
    return ''


def imaging_table(sheet, head_only):
    # The supplement holds findings + conclusion; the main export holds the report name. Merge by admission and time.
    s = supp[sheet].copy(); s['vis'] = s['科研就诊编号'].astype(str); s['time'] = pd.to_datetime(s['检查时间'], errors='coerce')
    s['text'] = s['检查所见'].fillna('').astype(str) + '。' + s['检查结论'].fillna('').astype(str)
    s['concl'] = s['检查结论'].fillna('').astype(str)
    m = sheets[sheet].copy(); m['vis'] = m['科研就诊编号'].astype(str); m['time'] = pd.to_datetime(m['检查时间'], errors='coerce')
    m['name'] = m['报告名称'].fillna('').astype(str) if '报告名称' in m else ''
    m['concl_main'] = m['检查结论'].fillna('').astype(str)
    x = s.merge(m[['vis', 'time', 'name']].drop_duplicates(['vis', 'time']), on=['vis', 'time'], how='left')
    only_main = m[~m.set_index(['vis', 'time']).index.isin(x.set_index(['vis', 'time']).index)].copy()
    only_main['text'] = only_main['concl_main']; only_main['concl'] = only_main['concl_main']
    x = pd.concat([x[['vis', 'time', 'name', 'text', 'concl']], only_main[['vis', 'time', 'name', 'text', 'concl']]], ignore_index=True)
    x['name'] = x['name'].fillna('')
    x = x.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x = x[(x.time >= x.admit - pd.Timedelta(days=1)) & (x.time <= x.discharge + pd.Timedelta(days=1))].copy()
    if head_only:
        x = x[x.name.str.contains(r'头|颅|脑', regex=True) | (x.name == '') & x.text.str.contains(r'脑室|脑沟|基底节|丘脑|小脑|中线结构')]
    x['day'] = (x.time - x.admit).dt.total_seconds() / 86400
    return x.sort_values(['vis', 'time']).reset_index(drop=True)


MRI = imaging_table('MRI报告', head_only=False)
MRI = MRI[MRI.text.str.contains(r'脑|丘脑|胼胝体|基底节|小脑|中线结构')].copy()
CT = imaging_table('CT报告', head_only=True)
FEATURES = {'thal_bilat': lambda t: find_thalamic(t, True), 'thal_any': lambda t: find_thalamic(t, False),
            'diffuse_edema': find_diffuse_edema, 'hemispheric': find_hemispheric}
for X in (MRI, CT):
    for k, f in FEATURES.items():
        X[k + '_txt'] = X.text.apply(f)
        X[k] = X[k + '_txt'].ne('')
    X['brainstem'] = X.text.apply(lambda t: bool(positive_clauses(t, r'脑干|桥脑|脑桥|延髓|中脑|大脑脚')))
for k, f in {'splenial': find_splenial, 'subcortical_dwi': find_subcortical_dwi, 'lobar_wm_dwi': lambda t: find_subcortical_dwi(t, LOBAR_WM)}.items():
    MRI[k + '_txt'] = MRI.text.apply(f)
    MRI[k] = MRI[k + '_txt'].ne('')
MRI['any_dwi'] = MRI.text.apply(lambda t: any(DWI_POS.search(c) and not NEG_IMG.search(c) for _, cls in clauses(t) for c in cls))
MRI['normal'] = MRI.concl.str.contains(r'未见明显异常|未见异常') & ~MRI[['thal_any', 'splenial', 'subcortical_dwi', 'diffuse_edema', 'any_dwi']].any(axis=1)
MRI['imp_ane'] = MRI.concl.str.contains(r'坏死性脑病') & ~MRI.concl.str.contains(r'不支持|排除|除外')
MRI['imp_mers'] = MRI.concl.str.contains(r'可逆性胼胝体|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病)')
MRI['imp_aesd'] = MRI.concl.str.contains(r'双相|双向|迟发性弥散|后期弥散|AESD')
MRI['imp_ane_txt'] = np.where(MRI.imp_ane, MRI.concl.str.replace(r'\s+', ' ', regex=True).str.slice(0, 120), '')


def per_admission(X, prefix, flags):
    out = X.groupby('vis').agg(**{f'{prefix}_n': ('text', 'size'), f'{prefix}_first_day': ('day', 'min')},
                               **{f'{prefix}_{c}': (c, 'any') for c in flags})
    for c in flags:
        if c + '_txt' in X:
            pos = X[X[c]]
            out[f'{prefix}_{c}_day'] = pos.groupby('vis').day.min()
            out[f'{prefix}_{c}_evidence'] = pos.groupby('vis').apply(lambda g: f"{prefix.upper()} day {g.day.iloc[0]:.0f}: {g[c + '_txt'].iloc[0][:160]}")
    return out


mri_v = per_admission(MRI, 'mri', ['thal_bilat', 'thal_any', 'brainstem', 'diffuse_edema', 'hemispheric', 'splenial', 'subcortical_dwi',
                                    'lobar_wm_dwi', 'any_dwi', 'normal', 'imp_ane', 'imp_mers', 'imp_aesd'])
for c in ['subcortical_dwi', 'lobar_wm_dwi']:
    mri_v[f'mri_{c}_day2plus'] = MRI[MRI.day >= 2].groupby('vis')[c].any()
    mri_v[f'mri_{c}_day2plus'] = mri_v[f'mri_{c}_day2plus'].fillna(False).astype(bool)
ct_v = per_admission(CT, 'ct', ['thal_bilat', 'thal_any', 'diffuse_edema', 'hemispheric'])
print('Index admissions with brain MRI:', mri_v.shape[0], '| with head CT:', ct_v.shape[0])
print({k: int(mri_v[k].sum()) for k in mri_v.columns if mri_v[k].dtype == bool})
print({k: int(ct_v[k].sum()) for k in ct_v.columns if ct_v[k].dtype == bool})

In [ ]:
# ITES evidence 2/4: EEG, CSF, fever, altered consciousness, seizures.
eeg = supp['脑电图检查'].copy(); eeg['vis'] = eeg['科研就诊编号'].astype(str)
eeg['text'] = (eeg['背景活动'].fillna('').astype(str) + '。' + eeg['印象'].fillna('').astype(str))
eeg = eeg[eeg.vis.isin(set(idx.vis))]
EEG_SLOW = re.compile(r'慢波活动|弥漫性慢波|广泛性慢波|弥漫性\s*\d|广泛性\s*\d|背景[^，,。；]{0,10}慢|慢化|δ波|δ活动|θ活动|θ波[^，,。；]{0,4}(?:增多|为主)'
                      r'|慢波为主|慢波增多|慢波较多|大量慢波|多量慢波|慢波明显')
EEG_SUPP = re.compile(r'电静息|暴发[-—]?抑制|爆发[-—]?抑制|低平抑制|抑制性脑波|低电压|电压低平|背景抑制')
EEG_SZ = re.compile(r'癫痫持续状态|电持续状态|非惊厥|NCSE|发作期')
eeg['slow'] = eeg.text.apply(lambda t: any(EEG_SLOW.search(c) and not re.search(r'未见|少量|无明显', c) for _, cls in clauses(t) for c in cls))
eeg['suppressed'] = eeg.text.str.contains(EEG_SUPP)
eeg['seizure_pattern'] = (eeg['印象'].fillna('').astype(str).str.contains(EEG_SZ)) | \
                         (eeg['发作期'].fillna('').astype(str).str.contains(r'监测到|发作') & ~eeg['发作期'].fillna('').astype(str).str.contains(r'未见特殊|未发生'))
eeg_v = eeg.groupby('vis').agg(eeg_n=('text', 'size'), eeg_slow=('slow', 'any'), eeg_suppressed=('suppressed', 'any'), eeg_seizure=('seizure_pattern', 'any'))

csf = pd.concat([sheets['实验室检查_脑脊液常规'], supp['实验室检查_脑脊液常规']]).drop_duplicates('报告编号')
csf['vis'] = csf['就诊编号'].astype(str); csf['wbc'] = parse_lab_value(csf['白细胞计数定量-定量结果'], qual_col(csf, '白细胞计数定量-定量结果'))[0]
csf_v = csf[csf.vis.isin(set(idx.vis))].groupby('vis').agg(csf_wbc_max=('wbc', 'max'))
cbio = pd.concat([sheets['实验室检查_脑脊液生化'], supp['实验室检查_脑脊液生化']]).drop_duplicates('报告编号')
cbio['vis'] = cbio['就诊编号'].astype(str)
prot = parse_lab_value(cbio['脑脊液蛋白定量-定量结果'], qual_col(cbio, '脑脊液蛋白定量-定量结果'))[0]
unit = cbio['脑脊液蛋白定量-定量结果单位'].astype(str).str.lower()
cbio['protein_gL'] = np.where(unit.str.contains('mg'), prot / 1000, prot)
csf_v = csf_v.join(cbio[cbio.vis.isin(set(idx.vis))].groupby('vis').protein_gL.max().rename('csf_protein_max'), how='outer')
vd = supp['实验室检查_脑脊液病毒DNA测定'].copy(); vd['vis'] = vd['就诊编号'].astype(str)
qual = [c for c in vd.columns if c.endswith('-定性结果')]
vd['positive'] = vd[qual].astype(str).apply(lambda r: r.str.contains('阳').any(), axis=1)
csf_v = csf_v.join(vd[vd.vis.isin(set(idx.vis))].groupby('vis').positive.any().rename('csf_virus_positive'), how='outer')

an = supp['儿科入院记录'].copy(); an['vis'] = an['科研就诊编号'].astype(str); an = an[an.vis.isin(set(idx.vis))]
an = an.groupby('vis').agg(cc=('主诉', lambda s: ' '.join(map(str, s.dropna()))), hpi=('现病史', lambda s: ' '.join(map(str, s.dropna()))),
                           pmh=('既往史', lambda s: ' '.join(map(str, s.dropna()))), pe=('体格检查', lambda s: ' '.join(map(str, s.dropna()))))
FEVER = re.compile(r'发热|发烧|高热|低热|中热|热峰|体温[^，,。；]{0,6}(?:升高|最高|达3[89]|达4)')
ENC = re.compile(r'昏迷|嗜睡|昏睡|意识障碍|意识不清|意识模糊|神志不清|神志模糊|神志恍惚|神志昏睡|神志嗜睡|神志浅昏迷|呼之不应|反应差|反应迟钝'
                 r'|反应低下|精神萎靡|精神反应差|精神差|谵妄|胡言乱语|烦躁不安|易激惹|GCS|格拉斯哥|淡漠|不认人|行为异常|精神行为异常|意识改变')
SEIZ = re.compile(r'抽搐|惊厥|抽动')
NEG_TXT = re.compile(r'无|不伴|否认|未|没有|不详')


def mentions(text, pat):
    return any(pat.search(c) and not NEG_TXT.search(c[:max(0, pat.search(c).start())][-6:]) for _, cls in clauses(text) for c in cls)


ev = idx[['vis', 'patient', 'subtype', 'age_y', 'admit', 'discharge', 'icu', 'death', 'poor', 'dx_text', 'course']].copy()
ev = ev.merge(an, left_on='vis', right_index=True, how='left')
for c in ['cc', 'hpi', 'pmh', 'pe']:
    ev[c] = ev[c].fillna('')
vit = supp['患者生命体征'].copy(); vit['vis'] = vit['科研就诊编号'].astype(str); vit = vit[vit.vis.isin(set(idx.vis))]
vit['T'] = pd.to_numeric(vit['体温(°C)'], errors='coerce'); vit['t'] = pd.to_datetime(vit['测量时间'], errors='coerce')
vit = vit.merge(idx[['vis', 'admit']], on='vis')
early = vit[(vit.t - vit.admit).dt.total_seconds().between(-86400, 3 * 86400)]
ev['tmax_72h'] = ev.vis.map(early.groupby('vis')['T'].max())
ev['fever_text'] = [mentions(a + '。' + b, FEVER) for a, b in zip(ev.cc, ev.hpi)]
ev['fever_course'] = ev.course.apply(lambda t: mentions(t[:400], FEVER))
ev['fever'] = ev.fever_text | ev.fever_course | (ev.tmax_72h >= 38.0)
ev['encephalopathy_text'] = [mentions(a + '。' + b + '。' + c, ENC) or mentions(d[:600], ENC) for a, b, c, d in zip(ev.cc, ev.hpi, ev.pe, ev.course)]
ev['seizure_text'] = [mentions(a + '。' + b, SEIZ) or mentions(c[:600], SEIZ) for a, b, c in zip(ev.cc, ev.hpi, ev.course)]
ev['biphasic_text'] = (ev.hpi + ev.course).str.contains(r'再次抽搐|再发抽搐|再次出现抽搐|又出现抽搐|再次惊厥|再次发作|双相|双向|第二次抽搐')
ev['prior_epilepsy'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'癫痫(?!持续)|婴儿痉挛'))) | \
                       ev.hpi.str.contains(r'诊断[“"]?癫痫|癫痫病史|既往.{0,8}癫痫')
ev['prior_neuro_disease'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'脑瘫|发育落后|发育迟缓|智力低下|遗传代谢|线粒体|脑积水')))
for t in (mri_v, ct_v, eeg_v, csf_v):
    ev = ev.merge(t, left_on='vis', right_index=True, how='left')
print('Fever documented:', int(ev.fever.sum()), '| altered consciousness documented:', int(ev.encephalopathy_text.sum()),
      '| EEG available:', int(ev.eeg_n.notna().sum()), '| CSF WBC available:', int(ev.csf_wbc_max.notna().sum()))

In [ ]:
# ITES evidence 3/4: shock and the six multi-organ items used for ASEM (first 72 h of admission).
def first72(sheet, col, how):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col + '-定量结果' in book[sheet].columns:
            d = book[sheet]
            parts.append(pd.DataFrame({'report': d['报告编号'].astype(str), 'vis': d['就诊编号'].astype(str),
                                       'v': parse_lab_value(d[col + '-定量结果'], qual_col(d, col + '-定量结果'))[0],
                                       'uln': pd.to_numeric(d.get(col + '-正常高值'), errors='coerce'),
                                       'lln': pd.to_numeric(d.get(col + '-正常低值'), errors='coerce'),
                                       't': pd.to_datetime(d['报告时间'], errors='coerce')}))
    x = pd.concat(parts).drop_duplicates('report').dropna(subset=['v'])
    x = x.merge(idx[['vis', 'admit']], on='vis')
    x = x[(x.t - x.admit).dt.total_seconds().between(-12 * 3600, 72 * 3600)]
    return x.groupby('vis').apply(how)


ev['hb_min'] = ev.vis.map(first72('实验室检查_血常规', '血红蛋白浓度定量', lambda g: g.v.min()))
ev['plt_min'] = ev.vis.map(first72('实验室检查_血常规', '血小板计数定量', lambda g: g.v.min()))
ev['inr_max'] = ev.vis.map(first72('实验室检查_凝血功能检查', '国际标准比率定量', lambda g: g.v.max()))
ev['fib_low'] = ev.vis.map(first72('实验室检查_凝血功能检查', '纤维蛋白原检测定量', lambda g: bool((g.v < g.lln).any())))
ev['ph_min'] = ev.vis.map(first72('实验室检查_血气分析', '酸碱度定量', lambda g: g.v.min()))
ev['be_min'] = ev.vis.map(first72('实验室检查_血气分析', '碱剩余定量', lambda g: g.v.min()))
ev['alt_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷丙转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['ast_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷草转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['cr_x_uln'] = ev.vis.map(first72('实验室检查_血液肾功能测定', '肌酐定量', lambda g: (g.v / g.uln).max()))
ev['ammonia_max'] = ev.vis.map(first72('实验室检查_血氨AMMO', '血氨AMMO定量', lambda g: g.v.max()))
ev['na_max'] = ev.vis.map(first72('实验室检查_血液电解质测定', '钠定量', lambda g: g.v.max()))
dx = ev.dx_text.fillna('')
MOF = {
    'anemia': ev.hb_min < 100,                                                         # Hb < 100 g/L
    'thrombocytopenia': ev.plt_min < 100,                                              # platelets < 100 x 10^9/L
    'dic': ((ev.inr_max >= 1.5) & ev.fib_low.fillna(False).astype(bool)) | dx.str.contains(r'弥散性血管内凝血|DIC'),
    'acidosis': (ev.ph_min < 7.30) | (ev.be_min <= -8),
    'liver': (ev.alt_x_uln > 2) | (ev.ast_x_uln > 2),                                   # > 2x age-specific ULN
    'renal': (ev.cr_x_uln > 1) | dx.str.contains(r'急性肾损伤|急性肾衰|急性肾功能'),        # > age-specific ULN
}
for k, v in MOF.items():
    ev[f'mof_{k}'] = v.fillna(False).astype(bool)
ev['mof_count'] = ev[[f'mof_{k}' for k in MOF]].sum(axis=1)
vaso72 = idx.set_index('vis').vaso_time.sub(idx.set_index('vis').admit).dt.total_seconds().le(72 * 3600)
ev['shock'] = ev.vis.map(vaso72).fillna(False).astype(bool) | dx.str.contains(r'休克') & ~dx.str.contains(r'失血性休克|心源性休克')
ev['septic_shock'] = dx.str.contains(r'脓毒性休克|感染性休克|脓毒症休克')
ev['sepsis_dx'] = dx.str.contains(r'脓毒症|败血症|脓毒性休克|感染性休克')
ev['cpr'] = ev.vis.map(idx.set_index('vis').cpr_any).fillna(False).astype(bool)
print('MOF items:', {k: int(ev[f'mof_{k}'].sum()) for k in MOF}, '| shock:', int(ev.shock.sum()), '| septic shock dx:', int(ev.septic_shock.sum()))

In [ ]:
# ITES evidence 4/4: alternative diagnoses that the consensus requires to be excluded.
EXCLUSIONS = {   # category: (pattern on discharge diagnoses of the index admission, applies to imaging-defined subtypes too)
    'CNS infection (bacterial/TB/fungal) or specific viral encephalitis': (r'化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|隐球菌|真菌性脑膜炎|脑脓肿|疱疹病毒性脑炎|单纯疱疹病毒性脑炎|单纯疱疹性脑炎|HSV脑炎|乙型脑炎|流行性乙型', True),
    'Autoimmune or demyelinating disease': (r'自身免疫性脑炎|自体免疫性脑炎|免疫介导性脑炎|NMDA|GABA|LGI1|MOG|ADEM|急性播散性脑脊髓炎|视神经脊髓炎|多发性硬化|桥本脑病', True),
    'Metabolic disease, Reye syndrome or IEM': (r'瑞氏|Reye|瑞夷|线粒体|Leigh|亚急性坏死性|甲基丙二酸|戊二酸|有机酸血症|尿素循环|遗传代谢|先天性代谢|代谢性脑病'
                                              r'|低血糖脑病|低血糖昏迷|肝性脑病|尿毒症|Wernicke|韦尼克|维生素B1缺乏', True),
    'Hypoxic-ischemic injury, drowning or asphyxia': (r'溺水|窒息|缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤', True),
    'Trauma': (r'颅脑损伤|颅脑外伤|脑挫|硬膜外血肿|外伤性|颅骨骨折|虐待', True),
    'Poisoning': (r'一氧化碳|药物中毒|农药|有机磷|毒蕈|蘑菇中毒|鼠药|酒精中毒|乙醇中毒|铅中毒|汞中毒|百草枯|药物过量', True),
    'Cerebrovascular disease or tumor': (r'脑梗|静脉窦血栓|烟雾病|颅内动脉瘤|动静脉畸形|血管炎|脑肿瘤|母细胞瘤|胶质瘤|室管膜瘤|生殖细胞瘤|颅内占位', True),
    'PRES or hypertensive encephalopathy': (r'可逆性后部|PRES|高血压脑病', True),
    'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)': (r'噬血|HLH|巨噬细胞活化|川崎|多系统炎症综合征|MIS-C|系统性红斑狼疮|狼疮', False),
    'Epilepsy or developmental and epileptic encephalopathy': (r'癫痫(?!持续)|婴儿痉挛|West综合征|Dravet|Lennox', False),
    'Malignancy or chemotherapy': (r'白血病|淋巴瘤|神经母细胞瘤|化疗', False),
}
UNCERTAIN = re.compile(r'[？?]|待排|待查|可能|不除外|疑似|可疑')
dx_items = ev.dx_text.fillna('').str.split(r'\s*\|\s*')


def excl_status(items, pat):
    # 'definite' when an item names the diagnosis without an uncertainty marker; 'possible' when only qualified mentions exist.
    hits = [x for x in items if re.search(pat, x)]
    if not hits:
        return ''
    return 'definite' if any(not UNCERTAIN.search(x) for x in hits) else 'possible'


for k, (p, _) in EXCLUSIONS.items():
    st_ = dx_items.apply(lambda L: excl_status(L, p))
    ev['excl: ' + k] = st_.eq('definite')
    ev['possible excl: ' + k] = st_.eq('possible')
ev['excl: Epilepsy or developmental and epileptic encephalopathy'] |= ev.prior_epilepsy
FIRES_LABEL = ev.subtype.eq('FIRES') | dx.str.contains(r'FIRES|热性感染相关|发热感染相关|感染相关性癫痫综合征')
ev.loc[FIRES_LABEL, 'excl: Epilepsy or developmental and epileptic encephalopathy'] = False   # epilepsy is the FIRES phenotype itself
ev['neonate'] = ev.age_y < 28 / 365
ev['csf_pleocytosis'] = ev.csf_wbc_max > 20
ev['csf_virus_positive'] = ev.csf_virus_positive.fillna(False).astype(bool)
HARD = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if hard]
SOFT = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if not hard]
print('Exclusion flags:', {k.replace('excl: ', ''): int(ev[k].sum()) for k in HARD + SOFT})
print('CSF WBC > 20 x10^6/L:', int(ev.csf_pleocytosis.sum()), '| CSF virus PCR positive:', int(ev.csf_virus_positive.sum()), '| neonates:', int(ev.neonate.sum()))

In [ ]:
# ITES assignment (hierarchical; every step is recorded in `ites_reason`).
def b(row, col):
    v = row.get(col)
    return bool(v) if pd.notna(v) else False


def assign(row):
    hard = [c.replace('excl: ', '') for c in HARD if row[c]]
    soft = [c.replace('excl: ', '') for c in SOFT if row[c]]
    thal = b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat') or b(row, 'mri_imp_ane')
    thal_day = np.nanmin([row.get('mri_thal_bilat_day', np.nan), row.get('ct_thal_bilat_day', np.nan), np.nan if not b(row, 'mri_imp_ane') else row.get('mri_first_day', np.nan)])
    edema = b(row, 'mri_diffuse_edema') or b(row, 'ct_diffuse_edema')
    mers_img = b(row, 'mri_splenial') or b(row, 'mri_imp_mers')
    aesd_img = b(row, 'mri_subcortical_dwi') or b(row, 'mri_imp_aesd') or (b(row, 'mri_lobar_wm_dwi_day2plus') and not mers_img) or \
        (row.subtype == 'AESD' and b(row, 'mri_any_dwi'))
    hemi = b(row, 'mri_hemispheric') or b(row, 'ct_hemispheric')
    has_neuroimaging = pd.notna(row.get('mri_n')) or pd.notna(row.get('ct_n'))
    fires = row.subtype == 'FIRES' or FIRES_LABEL.loc[row.name]
    flag = '' if row.fever else ' [fever not documented]'
    if row.neonate:
        return 'Non-ITES: neonatal encephalopathy', 'Neonate (<28 days)'
    if hard:
        return 'Non-ITES: alternative diagnosis', '; '.join(hard)
    if row.csf_virus_positive or row.csf_pleocytosis:
        return 'Non-ITES: encephalitis/meningitis', f'CSF WBC max {row.csf_wbc_max:g} x10^6/L' if row.csf_pleocytosis else 'CSF viral PCR positive'
    if thal and not (fires and not thal_day <= 7):
        return 'ANE', ('Bilateral thalamic lesions' if (b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat')) else 'Radiologist impression of ANE') + flag
    if fires:
        return 'FIRES', 'Clinician-diagnosed FIRES' + flag
    if edema:
        if row.septic_shock:
            return 'SAE', 'Diffuse cerebral edema with septic shock (septic shock excludes AFCE/ASEM)'
        if row.shock and row.mof_count >= 3:
            return 'ASEM', f'Diffuse cerebral edema, shock, {int(row.mof_count)} of 6 organ items' + flag
        return 'AFCE', 'Diffuse cerebral edema' + flag
    if aesd_img:
        late = b(row, 'mri_subcortical_dwi_day2plus') or b(row, 'mri_lobar_wm_dwi_day2plus')
        return 'AESD', 'White-matter restricted diffusion' + (' (MRI on day >= 2)' if late else ' (day of MRI < 2: check timing)') + flag
    if mers_img:
        if 'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)' in soft:
            return 'Non-ITES: alternative diagnosis', 'Splenial lesion with systemic inflammatory disease (MERS exclusion)'
        return 'MERS', 'Splenial lesion' + (' with restricted diffusion' if b(row, 'mri_any_dwi') else ' (diffusion not stated)') + flag
    if hemi and row.seizure_text:
        return 'HHE', 'Unilateral hemispheric swelling after seizures' + flag
    if row.subtype in ('ANE', 'MERS', 'AESD') and not has_neuroimaging:
        return row.subtype, 'Clinician-diagnosed; no neuroimaging report in the export' + flag
    if row.septic_shock or row.subtype == 'Septic_encephalopathy':
        return 'SAE', 'Septic shock or clinician-labeled septic encephalopathy, no ITES-specific imaging'
    if soft:
        return 'Non-ITES: alternative diagnosis', '; '.join(soft)
    if not row.fever:
        return 'Unclassifiable', 'No documented febrile illness'
    eeg_enceph = b(row, 'eeg_slow') or b(row, 'eeg_suppressed')
    if eeg_enceph and row.encephalopathy_text:
        return 'Possible ITES', 'Fever, altered consciousness, encephalopathic EEG, no syndrome-specific imaging'
    if eeg_enceph:
        return 'Unclassifiable', 'Fever and encephalopathic EEG, but altered consciousness not documented; no syndrome-specific imaging'
    if pd.isna(row.get('eeg_n')) and not has_neuroimaging:
        return 'Unclassifiable', 'No EEG and no neuroimaging'
    return 'Unclassifiable', 'No syndrome-specific imaging and no encephalopathic EEG documented'


ev[['ites_class', 'ites_reason']] = ev.apply(lambda r: pd.Series(assign(r)), axis=1)
TIER = {'ANE': 'Definite ITES', 'ASEM': 'Definite ITES', 'AFCE': 'Definite ITES', 'AESD': 'Definite ITES', 'MERS': 'Definite ITES',
        'HHE': 'ITES-related', 'FIRES': 'ITES-related', 'Possible ITES': 'Possible ITES', 'SAE': 'Sepsis-associated encephalopathy',
        'Unclassifiable': 'Unclassifiable'}
ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
ev['ites_algorithmic'] = ev.ites_class
CLIN = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'AESD', 'FIRES': 'FIRES', 'Septic_encephalopathy': 'SAE'}
ev['clinician_label'] = ev.subtype.map(CLIN).fillna('Other/unspecified')
specific_img = ev.ites_reason.str.contains(r'thalamic|edema|white-matter|Splenial|hemispheric', case=False)
conflict_excl = ev.ites_class.str.startswith('Non-ITES') & (ev.clinician_label.isin(['ANE', 'MERS', 'AESD', 'FIRES']) |
                 (ev.mri_thal_bilat.fillna(False).astype(bool) | ev.mri_splenial.fillna(False).astype(bool)))
possible_excl = ev[[c for c in ev.columns if c.startswith('possible excl: ')]].any(axis=1)
ev['review_reasons'] = [
    '; '.join(r for r, f in [
        ('clinician label differs from algorithm', cl not in ('Other/unspecified',) and cl != ic),
        ('exclusion conflicts with ITES-type imaging or clinician ITES label', ce),
        ('fever not documented', 'fever not documented' in rs),
        ('AESD imaging before day 2', 'check timing' in rs),
        ('splenial lesion without stated restricted diffusion', 'diffusion not stated' in rs),
        ('clinician label without neuroimaging in export', 'no neuroimaging report' in rs),
        ('qualified (?) alternative diagnosis present', pe and not ic.startswith('Non-ITES')),
        ('cardiac arrest/CPR documented', cp and ic in ('ANE', 'ASEM', 'AFCE', 'AESD')),
        ('altered consciousness not documented', (not en) and ic in ('ANE', 'ASEM', 'AFCE', 'AESD', 'MERS', 'HHE', 'FIRES')),
    ] if f)
    for cl, ic, ce, rs, pe, cp, en in zip(ev.clinician_label, ev.ites_class, conflict_excl, ev.ites_reason, possible_excl, ev.cpr, ev.encephalopathy_text)]
ev['needs_review'] = ev.review_reasons.ne('')
ADJ = HERE / 'output_ites' / 'ites_adjudicated.csv'
if ADJ.exists():
    adj = pd.read_csv(ADJ, dtype={'vis': str}).dropna(subset=['adjudicated_class'])
    ev = ev.merge(adj[['vis', 'adjudicated_class']], on='vis', how='left')
    ev['ites_class'] = ev.adjudicated_class.fillna(ev.ites_class)
    ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
    print('Adjudicated categories applied:', int(ev.adjudicated_class.notna().sum()))
else:
    print('No adjudication file found: algorithmic categories are used.')
print(ev.ites_class.value_counts().to_string())

In [ ]:
# Syndrome groups from the ITES re-classification (used in all tables, figures and models).
GROUP_ITES = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'Other ITES', 'AFCE': 'Other ITES', 'ASEM': 'Other ITES', 'HHE': 'Other ITES',
              'FIRES': 'Other ITES', 'Possible ITES': 'Possible ITES', 'SAE': 'SAE', 'Unclassifiable': 'Unclassifiable'}
ITES_COLS = ['vis', 'ites_class', 'ites_tier', 'ites_algorithmic', 'needs_review', 'mri_n', 'ct_n', 'eeg_n', 'mri_thal_bilat', 'ct_thal_bilat',
             'mri_brainstem', 'mri_diffuse_edema', 'ct_diffuse_edema', 'eeg_suppressed']
idx = idx.drop(columns=[c for c in ITES_COLS[1:] if c in idx.columns]).merge(ev[ITES_COLS], on='vis', how='left')
idx['group'] = idx.ites_class.map(GROUP_ITES).fillna('Non-ITES')
idx['group3'] = idx.group.where(idx.group.isin(['ANE', 'SAE']), 'Other')
idx['ites_any'] = idx.ites_tier.isin(['Definite ITES', 'ITES-related', 'Possible ITES'])
print(idx.group.value_counts().to_string())

In [ ]:
# Analytic populations and cohort flow.
icu = idx[idx.icu].copy()
P_TNT = icu[icu.tnt_early.notna() & ~icu.arrest_before_tnt].copy()
P_TNT['injury'] = P_TNT.tnt_early > HSCTN_URL
P_TNT['injury_alt'] = P_TNT.tnt_early > HSCTN_URL_ALT
P_TNT['log2_tnt'] = np.log2(P_TNT.tnt_early_ngL.clip(lower=1))
P_EF = icu[icu.lvef_early_min.notna() & ~icu.arrest_before_echo].copy()
P_EF['dysfunction'] = P_EF.lvef_early_min < 55
P_QT = icu[icu.qtcf_early.notna() & ~icu.arrest_before_ecg].copy()
P_QT['qt_prolonged'] = P_QT.qtcf_early >= 460
P_QT['qtb_prolonged'] = P_QT.qtcb_early >= 460
P_BNP = icu[icu.bnp_early.notna() & ~icu.arrest_before_bnp].copy()
P_BNP['log2_bnp'] = np.log2(P_BNP.bnp_early.clip(lower=1))

# Time anchoring at the first hs-cTn report: organ support already present vs started afterwards.
tt = P_TNT.tnt_early_time
P_TNT['vaso_before'] = P_TNT.vaso_time.notna() & (P_TNT.vaso_time <= tt)
P_TNT['vent_before'] = P_TNT.vent_time.notna() & (P_TNT.vent_time <= tt)
P_TNT['support_before'] = P_TNT.vaso_before | P_TNT.vent_before
P_TNT['vaso_after'] = P_TNT.vaso_time.notna() & (P_TNT.vaso_time > tt)
P_TNT['pressor_after'] = P_TNT.pressor_time.notna() & (P_TNT.pressor_time > tt)
P_TNT['vent_after'] = P_TNT.vent_time.notna() & (P_TNT.vent_time > tt)
P_DET = P_TNT[~P_TNT.support_before].copy()
WINDOW_H = 168   # primary definition: escalation within 7 days of the first hs-cTn report
def started_within(col, hours):
    h = (P_DET[col] - P_DET.tnt_early_time).dt.total_seconds() / 3600
    return (h > 0) & (h <= hours)
P_DET['vaso_7d'] = started_within('vaso_time', WINDOW_H)
P_DET['pressor_7d'] = started_within('pressor_time', WINDOW_H)
P_DET['vent_7d'] = started_within('vent_time', WINDOW_H)
hrs_to_discharge = (P_DET.discharge - P_DET.tnt_early_time).dt.total_seconds() / 3600   # death or critical discharge happens at discharge
P_DET['poor_7d'] = P_DET.poor.astype(bool) & (hrs_to_discharge <= WINDOW_H)
P_DET['poor_72h'] = P_DET.poor.astype(bool) & (hrs_to_discharge <= 72)
P_DET['deterioration'] = P_DET.vaso_7d | P_DET.vent_7d | P_DET.poor_7d
P_DET['deterioration_72h'] = started_within('vaso_time', 72) | started_within('vent_time', 72) | P_DET.poor_72h
P_DET['deterioration_any'] = P_DET.vaso_after | P_DET.vent_after | P_DET.poor.astype(bool)
esc = pd.concat([P_DET.vaso_time.where(P_DET.vaso_after), P_DET.vent_time.where(P_DET.vent_after)], axis=1).min(axis=1)
P_DET['hours_to_escalation'] = (esc - P_DET.tnt_early_time).dt.total_seconds() / 3600

flow = pd.DataFrame([
    ('AES patients (any admission)', len(coh)),
    ('Acute-phase index admission', int(coh.acute_index.sum())),
    ('Index admission with recorded disposition (analysis base)', len(idx)),
    ('ICU stay during index admission', len(icu)),
    ('ICU: hs-cTn within 48 h of admission', int(icu.tnt_early.notna().sum())),
    ('ICU: hs-cTn excluded for cardiac arrest before or on the day of measurement', int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum())),
    ('Primary hs-cTn cohort', len(P_TNT)),
    ('LVEF cohort (echo within 72 h, arrest rule applied)', len(P_EF)),
    ('QTc cohort (ECG within 48 h, arrest rule applied)', len(P_QT)),
    ('NT-proBNP cohort (within 48 h, arrest rule applied)', len(P_BNP)),
    ('Primary cohort without organ support at the first hs-cTn report (deterioration cohort)', len(P_DET)),
], columns=['step', 'n'])
flow['events (death or critical AMA)'] = [np.nan, np.nan, int(idx.poor.sum()), int(icu.poor.sum()),
    int(icu[icu.tnt_early.notna()].poor.sum()), np.nan, int(P_TNT.poor.sum()), int(P_EF.poor.sum()),
    int(P_QT.poor.sum()), int(P_BNP.poor.sum()), int(P_DET.poor.sum())]
flow['events (deterioration within 7 days)'] = [np.nan] * 10 + [int(P_DET.deterioration.sum())]
flow

In [ ]:
# Table 1: primary hs-cTn cohort by myocardial injury (early hs-cTn above the URL).
def med_iqr(s, d=1):
    s = s.dropna()
    return f'{rnd(s.median(), d)} ({rnd(s.quantile(.25), d)}-{rnd(s.quantile(.75), d)})' if len(s) else ''

def n_pct(s):
    s = s.dropna().astype(bool)
    return f'{int(s.sum())} ({rnd(100 * s.mean())})' if len(s) else ''

def p_cont(a, b):
    a, b = a.dropna(), b.dropna()
    return stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan

def p_bin(a, b):
    t = np.array([[a.sum(), (~a).sum()], [b.sum(), (~b).sum()]])
    return stats.fisher_exact(t)[1] if t.min() < 5 else stats.chi2_contingency(t)[1]

def table_by(df, flag, cont, binary, cats):
    g1, g0 = df[df[flag]], df[~df[flag]]
    rows = [('N', str(len(df)), str(len(g1)), str(len(g0)), np.nan)]
    for item in cont:
        lab, col, dec = item if len(item) == 3 else (*item, 1)   # decimals shown for this variable
        rows.append((lab + ', median (IQR)', med_iqr(df[col], dec), med_iqr(g1[col], dec), med_iqr(g0[col], dec), p_cont(g1[col], g0[col])))
    for lab, col in binary:
        rows.append((lab + ', n (%)', n_pct(df[col]), n_pct(g1[col]), n_pct(g0[col]), p_bin(g1[col].astype(bool), g0[col].astype(bool))))
    for lab, col in cats:
        cs, c1, c0 = df[col].astype(str), g1[col].astype(str), g0[col].astype(str)
        p = stats.chi2_contingency(pd.crosstab(cs, df[flag]))[1]
        rows.append((lab, '', '', '', p))
        for lvl in cs.value_counts().index:
            rows.append(('   ' + str(lvl), n_pct(cs == lvl), n_pct(c1 == lvl), n_pct(c0 == lvl), np.nan))
    return pd.DataFrame(rows, columns=['Characteristic', 'All', 'Exposed', 'Not exposed', 'P'])

CONT = [('Age, years', 'age_y', 1), ('Early hs-cTn, ng/L', 'tnt_early_ngL', 1), ('Early lactate, mmol/L', 'lac_early', 1),
        ('Early NT-proBNP, pg/mL', 'bnp_early', 0), ('LVEF (minimum within 72 h), %', 'lvef_early_min', 0), ('QTc Fridericia (first 48 h), ms', 'qtcf_early', 0)]
BIN = [('Male', 'male'), ('Age < 1 year', 'age_lt1'), ('Invasive ventilation', 'ventilation'), ('Vasoactive infusion', 'vasoactive'),
       ('Status epilepticus', 'status_epilepticus'), ('Septic shock (diagnosis)', 'septic_shock'), ('Lactate >= 4 mmol/L (first 48 h)', 'lactate_ge4'),
       ('Methylprednisolone', 'methylprednisolone'), ('IVIG', 'ivig'), ('Tocilizumab', 'tocilizumab'),
       ('In-hospital death', 'death'), ('Critical discharge against advice', 'critical_ama'), ('Death or critical discharge against advice', 'poor')]
CATS = [('Syndrome group (ITES re-classification)', 'group'), ('Era', 'era')]
table1 = table_by(P_TNT, 'injury', CONT, BIN, CATS).rename(columns={'Exposed': f'hs-cTn > {URL_NGL} ng/L', 'Not exposed': f'hs-cTn <= {URL_NGL} ng/L'})
table1

In [ ]:
# Supplementary: who was tested? ICU index admissions with versus without early hs-cTn.
icu['tested'] = icu.tnt_early.notna()
tab_tested = table_by(icu, 'tested', [('Age, years', 'age_y')],
                      [('Male', 'male'), ('Invasive ventilation', 'ventilation'), ('Vasoactive infusion', 'vasoactive'),
                       ('Status epilepticus', 'status_epilepticus'), ('Death or critical discharge against advice', 'poor')],
                      [('Syndrome group (ITES re-classification)', 'group')]).rename(columns={'Exposed': 'Tested', 'Not exposed': 'Not tested'})
tab_tested

In [ ]:
# Table 2: cardiac phenotype across syndrome groups (ICU index admissions; arrest rule applied per measure).
ORDER_G = ['ANE', 'MERS', 'Other ITES', 'Possible ITES', 'SAE', 'Non-ITES', 'Unclassifiable']
def frac(df, col):
    s = df[col].dropna().astype(bool)
    return int(s.sum()), len(s)
rows = []
for g in [g for g in ORDER_G if (icu.group == g).sum() > 0]:
    a, b, c, d = P_TNT[P_TNT.group == g], P_EF[P_EF.group == g], P_QT[P_QT.group == g], P_BNP[P_BNP.group == g]
    i1, n1 = frac(a, 'injury'); i2, n2 = frac(b, 'dysfunction'); i3, n3 = frac(c, 'qt_prolonged'); i4, n4 = frac(c, 'qtb_prolonged')
    rows.append({'group': g, 'ICU N': int((icu.group == g).sum()),
                 f'hs-cTn > {URL_NGL} ng/L': f'{i1}/{n1} ({rnd(100 * i1 / n1, 0)}%)' if n1 else '',
                 'hs-cTn median (IQR), ng/L': med_iqr(a.tnt_early_ngL),
                 'LVEF < 55%': f'{i2}/{n2} ({rnd(100 * i2 / n2, 0)}%)' if n2 else '',
                 'QTcF >= 460 ms': f'{i3}/{n3} ({rnd(100 * i3 / n3, 0)}%)' if n3 else '',
                 'QTcB >= 460 ms': f'{i4}/{n4} ({rnd(100 * i4 / n4, 0)}%)' if n4 else '',
                 'NT-proBNP median (IQR), pg/mL': med_iqr(d.bnp_early, 0),
                 '_inj': (i1, n1), '_dys': (i2, n2), '_qt': (i3, n3), '_qtb': (i4, n4)})
pheno = pd.DataFrame(rows)

def mc_chi2_p(groups, flags, reps=10000, seed=SEED):
    # Monte Carlo (permutation) p value for a k x 2 chi-square test; robust to sparse cells.
    codes = pd.Categorical(groups).codes; f = np.asarray(flags, float)
    def stat(ff):
        k = codes.max() + 1
        n = np.bincount(codes, minlength=k); n1 = np.bincount(codes, weights=ff, minlength=k)
        p = ff.mean(); e1, e0 = n * p, n * (1 - p)
        return np.sum((n1 - e1) ** 2 / e1 + ((n - n1) - e0) ** 2 / e0)
    obs = stat(f); r = np.random.default_rng(seed)
    hits = sum(stat(r.permutation(f)) >= obs - 1e-12 for _ in range(reps))
    return (hits + 1) / (reps + 1)

p_inj = stats.chi2_contingency(pd.crosstab(P_TNT.group, P_TNT.injury))[1]
p_tnt = stats.kruskal(*[P_TNT[P_TNT.group == g].tnt_early_ngL for g in ORDER_G if (P_TNT.group == g).sum() > 0]).pvalue
p_bnp = stats.kruskal(*[P_BNP[P_BNP.group == g].bnp_early.dropna() for g in ORDER_G if P_BNP[P_BNP.group == g].bnp_early.notna().sum() > 0]).pvalue
p_dys = stats.chi2_contingency(pd.crosstab(P_EF.group, P_EF.dysfunction))[1]
p_qt = stats.chi2_contingency(pd.crosstab(P_QT.group, P_QT.qt_prolonged))[1]
p_qtb = stats.chi2_contingency(pd.crosstab(P_QT.group, P_QT.qtb_prolonged))[1]
p_mc = {'injury': mc_chi2_p(P_TNT.group, P_TNT.injury), 'lvef': mc_chi2_p(P_EF.group, P_EF.dysfunction),
        'qtc': mc_chi2_p(P_QT.group, P_QT.qt_prolonged), 'qtcb': mc_chi2_p(P_QT.group, P_QT.qtb_prolonged)}
print(f'P across groups: injury {p_inj:.2g}; hs-cTn {p_tnt:.2g}; NT-proBNP {p_bnp:.2g}; LVEF<55 {p_dys:.2g}; QTcF>=460 {p_qt:.2g}; QTcB>=460 {p_qtb:.2g}')
print('Monte Carlo P (10,000 permutations):', {k: round(v, 4) for k, v in p_mc.items()})
table2 = pheno.drop(columns=['_inj', '_dys', '_qt', '_qtb'])
table2

In [ ]:
# Figure helpers (submission style: no titles, panel letters aligned with the y-axis title).
# Color figures (PCCM encourages color): validated categorical slots 1-2 (blue, orange; adjacent CVD and normal-vision
# checks pass on white) plus a neutral gray for "other"; every mark is opaque so that EPS export keeps it intact.
GREY_DARK, GREY_MID, GREY_LIGHT = '#404040', '#8c8c8c', '#d0d0d0'
BLUE, ORANGE, NEUTRAL = '#2a78d6', '#eb6834', '#898781'
BLUE_BAND = '#b7d3f6'                 # light step of the blue ramp, for confidence bands
REF_INK = '#52514e'                   # reference lines
TIER_COLOR = {'ANE': BLUE, 'MERS': BLUE, 'Other ITES': BLUE, 'Possible ITES': BLUE, 'SAE': ORANGE,
              'Non-ITES': NEUTRAL, 'Unclassifiable': NEUTRAL}
TIER_HANDLES = [Patch(facecolor=BLUE, label='ITES (definite, related, or possible)'),
                Patch(facecolor=ORANGE, label='Sepsis-associated encephalopathy'),
                Patch(facecolor=NEUTRAL, label='Non-ITES or unclassifiable')]

def panel_letters(fig, axes, letters):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    for ax, letter in zip(axes, letters):
        bbox = ax.yaxis.label.get_window_extent(renderer=renderer)
        x0 = inv.transform((bbox.x0, 0))[0]
        top = ax.get_position().y1
        fig.text(x0, top + 0.012, letter, fontsize=18, fontweight='bold', ha='left', va='bottom')

def save(fig, name):
    # PNG for previews, PDF and EPS (vector) for submission; journals accept EPS for line art.
    fig.savefig(FIG / f'{name}.png', dpi=600, bbox_inches='tight')
    fig.savefig(FIG / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG / f'{name}.eps', bbox_inches='tight')

def bar_fraction(ax, labels, fracs, ylabel, color=BLUE):
    pct = [100 * a / b if b else 0 for a, b in fracs]
    top = max(pct + [10])
    ymax = 112 if top >= 90 else top * 1.25
    colors = [TIER_COLOR.get(l, color) for l in labels]
    bars = ax.bar(range(len(labels)), pct, color=colors, edgecolor='none', width=0.65)
    for i, (a, b) in enumerate(fracs):
        ax.text(i, pct[i] + 0.015 * ymax, f'{a}/{b}', ha='center', va='bottom', fontsize=11)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel(ylabel); ax.set_ylim(0, ymax)
    if top >= 90:
        ax.set_yticks(range(0, 101, 20))
    return bars

def plain_log_ticks(axis, ticks, lo, hi):
    ticks = [t for t in ticks if lo <= t <= hi]
    axis.set_major_locator(FixedLocator(ticks))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    axis.set_minor_locator(NullLocator())

In [ ]:
# Figure 1: cardiac phenotype across syndrome groups.
fig, axs = plt.subplots(2, 2, figsize=(12, 9.5))
fig.subplots_adjust(hspace=0.55, wspace=0.32)
groups = [g for g in ORDER_G if (P_TNT.group == g).sum() > 0]
bar_fraction(axs[0, 0], groups, [tuple(pheno.set_index('group').loc[g, '_inj']) for g in groups], f'hs-cTnT > {URL_NGL} ng/L (%)')

ax = axs[0, 1]
data = [P_TNT[P_TNT.group == g].tnt_early_ngL.clip(lower=1).values for g in groups]
for i, (g, d) in enumerate(zip(groups, data)):
    ax.scatter(i + rng.uniform(-0.18, 0.18, len(d)), d, s=9, color=TIER_COLOR[g], linewidths=0, zorder=2)
ax.boxplot(data, positions=range(len(groups)), widths=0.55, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='none', edgecolor='black'), medianprops=dict(color='black', linewidth=1.8),
           whiskerprops=dict(color='black'), capprops=dict(color='black'), zorder=3)
ax.set_yscale('log')
ylo, yhi = ax.get_ylim()
plain_log_ticks(ax.yaxis, [1, 3, 10, 30, 100, 300, 1000, 3000], ylo, yhi)
ref = ax.axhline(URL_NGL, color=REF_INK, linestyle='--', linewidth=1.1, zorder=1)
ax.set_xticks(range(len(groups))); ax.set_xticklabels(groups, rotation=30, ha='right')
ax.set_ylabel('hs-cTnT (ng/L)')
ax.legend([ref], [f'Upper reference limit ({URL_NGL} ng/L)'], loc='upper right')

gq = [g for g in ORDER_G if (P_EF.group == g).sum() > 0]
bar_fraction(axs[1, 0], gq, [tuple(pheno.set_index('group').loc[g, '_dys']) for g in gq], 'LVEF < 55% (%)')
gq = [g for g in ORDER_G if (P_QT.group == g).sum() > 0]
bar_fraction(axs[1, 1], gq, [tuple(pheno.set_index('group').loc[g, '_qt']) for g in gq], 'QTc (Fridericia) ≥ 460 ms (%)')
fig.legend(handles=TIER_HANDLES, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.04), fontsize=12)
panel_letters(fig, [axs[0, 0], axs[0, 1], axs[1, 0], axs[1, 1]], 'ABCD')
save(fig, 'Figure1_cardiac_phenotype_by_syndrome')
plt.show()

In [ ]:
# Association with death or critical discharge against advice: modified Poisson regression (robust SE).
def mpoisson(df, formula):
    m = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(cov_type='HC0')
    return m

def rr_row(m, term):
    b, se = m.params[term], m.bse[term]
    return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), m.pvalues[term]

for d in (P_TNT, P_EF, P_QT, P_BNP):
    for c in ['poor', 'male', 'age_lt1', 'ventilation', 'vasoactive', 'status_epilepticus']:
        d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int)
    d['group_sae'] = (d.group3 == 'SAE').astype(int)

M1 = ' + age_y + male'
M2 = M1 + ' + group_ane + group_sae'
M3 = M2 + ' + ventilation + vasoactive + status_epilepticus'
EXPOSURES = [
    (f'hs-cTn > {URL_NGL} ng/L', P_TNT, 'injury'), ('hs-cTn, per doubling', P_TNT, 'log2_tnt'),
    ('LVEF < 55%', P_EF, 'dysfunction'), ('NT-proBNP, per doubling', P_BNP, 'log2_bnp'),
    ('QTc (Fridericia) >= 460 ms', P_QT, 'qt_prolonged'),
]
res = []
for lab, d, var in EXPOSURES:
    d = d.copy(); d[var] = d[var].astype(float)
    row = {'exposure': lab, 'N': len(d), 'events': int(d.poor.sum())}
    if d[var].nunique() == 2:
        e1, n1 = int(d[d[var] == 1].poor.sum()), int((d[var] == 1).sum())
        e0, n0 = int(d[d[var] == 0].poor.sum()), int((d[var] == 0).sum())
        row['events/N exposed'] = f'{e1}/{n1}'; row['events/N unexposed'] = f'{e0}/{n0}'
    for mname, rhs in [('Model 1', M1), ('Model 2', M2), ('Model 3', M3)]:
        try:
            m = mpoisson(d, f'poor ~ {var}{rhs}')
            rr, lo, hi, p = rr_row(m, var)
            row[mname] = f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})'; row[mname + ' P'] = p
            row[mname + '_num'] = (rr, lo, hi)
        except Exception as e:
            row[mname] = f'not estimable ({type(e).__name__})'
    res.append(row)
assoc = pd.DataFrame(res)
table3 = assoc[[c for c in assoc.columns if not c.endswith('_num')]]
table3

In [ ]:
# Overlap-weighted estimate for myocardial injury (propensity model carries the covariates; outcome model does not).
PS_COVS = ['age_y', 'age_lt1', 'male', 'group_ane', 'group_sae', 'ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock_i', 'lac_ge4_i', 'lac_missing']
def prep_ps(d):
    d = d.copy()
    d['septic_shock_i'] = d.septic_shock.astype(int)
    d['lac_missing'] = d.lac_early.isna().astype(int)
    d['lac_ge4_i'] = (d.lac_early >= 4).fillna(False).astype(int)
    d['injury_i'] = d.injury.astype(int)
    return d

def overlap_rr(d):
    X = sm.add_constant(d[PS_COVS].astype(float))
    ps = sm.Logit(d.injury_i, X).fit(disp=0, method='bfgs', maxiter=500).predict(X)
    w = np.where(d.injury_i == 1, 1 - ps, ps)
    r1 = np.average(d.poor[d.injury_i == 1], weights=w[d.injury_i == 1])
    r0 = np.average(d.poor[d.injury_i == 0], weights=w[d.injury_i == 0])
    return r1, r0, w, ps

def smd(d, w, col):
    a, b = d[d.injury_i == 1], d[d.injury_i == 0]
    wa, wb = (None, None) if w is None else (w[d.injury_i == 1], w[d.injury_i == 0])
    ma, mb = np.average(a[col], weights=wa), np.average(b[col], weights=wb)
    va = np.average((a[col] - ma) ** 2, weights=wa); vb = np.average((b[col] - mb) ** 2, weights=wb)
    return (ma - mb) / np.sqrt((va + vb) / 2) if (va + vb) > 0 else 0.0

D = prep_ps(P_TNT)
r1, r0, w, ps = overlap_rr(D)
boot = []
for _ in range(1000):
    s = D.sample(len(D), replace=True, random_state=int(rng.integers(1e9)))
    if s.injury_i.nunique() < 2 or s.poor.sum() == 0:
        continue
    try:
        b1, b0, _, _ = overlap_rr(s)
        if b0 > 0:
            boot.append((b1 / b0, b1 - b0))
    except Exception:
        pass
boot = np.array(boot)
ow = {'risk_injury': r1, 'risk_no_injury': r0, 'RR': r1 / r0, 'RR_lo': np.percentile(boot[:, 0], 2.5), 'RR_hi': np.percentile(boot[:, 0], 97.5),
      'RD': r1 - r0, 'RD_lo': np.percentile(boot[:, 1], 2.5), 'RD_hi': np.percentile(boot[:, 1], 97.5), 'bootstrap_reps': len(boot)}
balance = pd.DataFrame({'covariate': PS_COVS, 'SMD unweighted': [smd(D, None, c) for c in PS_COVS],
                        'SMD overlap-weighted': [smd(D, w, c) for c in PS_COVS]}).round(3)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in ow.items()})
balance

In [ ]:
# Time-anchored analyses at the first hs-cTn report.
def prep_det(d):
    d = d.copy()
    for c in ['poor', 'male', 'status_epilepticus', 'injury', 'injury_alt', 'vaso_before', 'vent_before', 'deterioration', 'deterioration_72h', 'deterioration_any']:
        if c in d:
            d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int)
    d['group_sae'] = (d.group3 == 'SAE').astype(int)
    d['log2_tnt'] = np.log2(d.tnt_early_ngL.clip(lower=1))
    d['lac_ge4'] = (d.lac_early >= 4).fillna(False).astype(int)
    d['lac_missing'] = d.lac_early.isna().astype(int)
    return d

BASE_DET = ' + age_y + male + group_ane + group_sae + status_epilepticus'
A_ = prep_det(P_TNT)
rr_pre = rr_row(mpoisson(A_, 'poor ~ injury' + BASE_DET + ' + vaso_before + vent_before'), 'injury')
Dt = prep_det(P_DET)
Dt['det_pressor'] = (P_DET.pressor_7d | P_DET.vent_7d | P_DET.poor_7d).astype(int)
Dt['det_no_vent'] = (P_DET.vaso_7d | P_DET.poor_7d).astype(int)

def det_row(label, d, outcome, var, rhs):
    m = mpoisson(d, f'{outcome} ~ {var}{rhs}')
    rr, lo, hi, p = rr_row(m, var)
    row = {'analysis': label, 'N': len(d), 'events': int(d[outcome].sum())}
    if d[var].nunique() == 2:
        row['events/N, injury'] = f'{int(d[d[var] == 1][outcome].sum())}/{int((d[var] == 1).sum())}'
        row['events/N, no injury'] = f'{int(d[d[var] == 0][outcome].sum())}/{int((d[var] == 0).sum())}'
    row.update({'RR (95% CI)': f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})', 'P': p, '_num': (rr, lo, hi)})
    return row

t5 = [
    {'analysis': 'Death or critical AMA, adjusted for organ support present before testing', 'N': len(A_), 'events': int(A_.poor.sum()),
     'RR (95% CI)': f'{rnd(rr_pre[0], 2)} ({rnd(rr_pre[1], 2)}-{rnd(rr_pre[2], 2)})', 'P': rr_pre[3], '_num': rr_pre[:3]},
    det_row('Subsequent deterioration within 7 days (primary definition)', Dt, 'deterioration', 'injury', BASE_DET),
    det_row('Subsequent deterioration, additionally adjusted for lactate', Dt, 'deterioration', 'injury', BASE_DET + ' + lac_ge4 + lac_missing'),
    det_row('Subsequent deterioration, hs-cTn per doubling', Dt, 'deterioration', 'log2_tnt', BASE_DET),
    det_row('Deterioration defined with vasopressors only (dobutamine/milrinone excluded)', Dt, 'det_pressor', 'injury', BASE_DET),
    det_row('Deterioration without ventilation (vasoactive start or death/critical AMA)', Dt, 'det_no_vent', 'injury', BASE_DET),
    det_row('Subsequent deterioration, age >= 1 year', Dt[Dt.age_y >= 1], 'deterioration', 'injury', BASE_DET),
    det_row('Deterioration with escalation within 72 h', Dt, 'deterioration_72h', 'injury', BASE_DET),
    det_row('Deterioration with escalation at any time during admission', Dt, 'deterioration_any', 'injury', BASE_DET),
    det_row(f'Subsequent deterioration, injury defined as hs-cTn > {ALT_NGL} ng/L', Dt, 'deterioration', 'injury_alt', BASE_DET),
    det_row('Subsequent deterioration, consensus ITES only (definite, related or possible)', Dt[Dt.ites_any.astype(bool)], 'deterioration', 'injury', BASE_DET.replace(' + group_sae', '')),
    det_row('Subsequent deterioration, excluding non-ITES and unclassifiable', Dt[Dt.ites_any.astype(bool) | Dt.group.eq('SAE')], 'deterioration', 'injury', BASE_DET),
]
table5 = pd.DataFrame(t5)
# Effect modification by consensus ITES status (definite, related or possible ITES vs all other diagnoses).
Dt['ites_i'] = Dt.ites_any.astype(int)
m_int = mpoisson(Dt, 'deterioration ~ injury * ites_i' + BASE_DET)
p_interaction = float(m_int.pvalues['injury:ites_i'])
sub_rows = []
for lab, d in [('Consensus ITES (definite, related or possible)', Dt[Dt.ites_i == 1]),
               ('Other diagnoses (SAE, non-ITES, unclassifiable)', Dt[Dt.ites_i == 0]), ('All', Dt)]:
    e1, n1 = int(d[d.injury == 1].deterioration.sum()), int((d.injury == 1).sum())
    e0, n0 = int(d[d.injury == 0].deterioration.sum()), int((d.injury == 0).sum())
    sub_rows.append({'subgroup': lab, 'N': len(d), 'deterioration, injury': f'{e1}/{n1} ({rnd(100 * e1 / n1)}%)',
                     'deterioration, no injury': f'{e0}/{n0} ({rnd(100 * e0 / n0)}%)',
                     'unadjusted RR': float(rnd((e1 / n1) / (e0 / n0), 2)) if e0 else np.nan,
                     'risk difference, %': float(rnd(100 * (e1 / n1 - e0 / n0), 1))})
det_subgroups = pd.DataFrame(sub_rows)
# Fragility index (unadjusted Fisher exact test): children in the lower-risk group whose outcome would have to change
# from no event to event before the association lost statistical significance (p >= 0.05).
def fragility_index(e1, n1, e0, n0, alpha=0.05):
    tab = lambda a, b: [[a, n1 - a], [b, n0 - b]]
    p0 = stats.fisher_exact(tab(e1, e0))[1]
    if p0 >= alpha:
        return 0, p0
    # Events are added to the lower-risk group (Walsh et al.; with unequal group sizes, the group with the lower event rate).
    fi, add_to_0 = 0, e0 / n0 <= e1 / n1
    while True:
        if (add_to_0 and e0 >= n0) or (not add_to_0 and e1 >= n1):
            return np.nan, p0
        if add_to_0:
            e0 += 1
        else:
            e1 += 1
        fi += 1
        if stats.fisher_exact(tab(e1, e0))[1] >= alpha:
            return fi, p0
fi_e1, fi_n1 = int(Dt[Dt.injury == 1].deterioration.sum()), int((Dt.injury == 1).sum())
fi_e0, fi_n0 = int(Dt[Dt.injury == 0].deterioration.sum()), int((Dt.injury == 0).sum())
frag_index, frag_fisher_p = fragility_index(fi_e1, fi_n1, fi_e0, fi_n0)
print(f'Fragility index (deterioration, Fisher exact): {frag_index} (unadjusted Fisher p = {frag_fisher_p:.4f})')
print(f'P for interaction (myocardial injury x consensus ITES): {p_interaction:.3f}')
print(det_subgroups.to_string(index=False))
h_inj = P_DET[P_DET.injury].hours_to_escalation.dropna(); h_no = P_DET[~P_DET.injury].hours_to_escalation.dropna()
print('Hours from first hs-cTn report to escalation: median (IQR)', f'{P_DET.hours_to_escalation.median():.1f}',
      P_DET.hours_to_escalation.quantile([.25, .75]).round(1).tolist(), '| n =', int(P_DET.hours_to_escalation.notna().sum()))
print('Escalation within 7 days: vasoactive', int(P_DET.vaso_7d.sum()), '| ventilation', int(P_DET.vent_7d.sum()),
      '| death or critical AMA within 7 days', int(P_DET.poor_7d.sum()), '| during the admission', int(P_DET.poor.sum()))
table5.drop(columns=['_num'])

In [ ]:
# Supplementary: deterioration cohort (no organ support at the first hs-cTn report) by myocardial injury.
table_det = table_by(P_DET, 'injury', CONT, BIN, CATS).rename(columns={'Exposed': f'hs-cTn > {URL_NGL} ng/L', 'Not exposed': f'hs-cTn <= {URL_NGL} ng/L'})
table_det

In [ ]:
# Dose-response: event rates across early hs-cTn categories (multiples of the URL) and age-adjusted restricted cubic splines.
bins = [0, URL_NGL, 5 * URL_NGL, 10 * URL_NGL, np.inf]
labels_c = [f'≤{URL_NGL}', f'{URL_NGL + 1}–{5 * URL_NGL}', f'{5 * URL_NGL + 1}–{10 * URL_NGL}', f'>{10 * URL_NGL}']

def trend_table(d, outcome):
    d = d.copy()
    d['cat'] = pd.cut(d.tnt_early_ngL, bins=bins, labels=labels_c, right=True)
    t = d.groupby('cat', observed=False).agg(N=(outcome, 'size'), events=(outcome, 'sum'))
    t['rate %'] = [float(rnd(100 * e / n, 1)) if n else np.nan for e, n in zip(t.events, t.N)]
    k = np.arange(len(t)); n = t.N.values.astype(float); e = t.events.values.astype(float); pbar = e.sum() / n.sum()
    T = np.sum(k * (e - n * pbar)); V = pbar * (1 - pbar) * (np.sum(n * k ** 2) - np.sum(n * k) ** 2 / n.sum())
    return t, 2 * (1 - stats.norm.cdf(abs(T / np.sqrt(V))))   # Cochran-Armitage test for trend

dose, p_trend = trend_table(prep_det(P_TNT), 'poor')
dose_det, p_trend_det = trend_table(Dt, 'deterioration')
print(dose, f'\nP for trend (death or critical AMA): {p_trend:.3g}\n')
print(dose_det, f'\nP for trend (subsequent deterioration): {p_trend_det:.3g}')

def rcs3(x, knots):
    t1, t2, t3 = knots
    pos = lambda u: np.clip(u, 0, None) ** 3
    return (pos(x - t1) - pos(x - t2) * (t3 - t1) / (t3 - t2) + pos(x - t3) * (t2 - t1) / (t3 - t2)) / (t3 - t1) ** 2

def spline_fit(d, outcome):
    x = np.log2(d.tnt_early_ngL.clip(lower=1)).values
    knots = np.percentile(x, [10, 50, 90])
    def design(xx, age):
        xx = np.asarray(xx, float)
        return np.column_stack([np.ones_like(xx), xx, rcs3(xx, knots), np.asarray(age, float) * np.ones_like(xx)])
    X = design(x, d.age_y.values); y = d[outcome].astype(float).values
    spl = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    lin = sm.GLM(y, X[:, [0, 1, 3]], family=sm.families.Binomial()).fit()
    p_nl = stats.chi2.sf(2 * (spl.llf - lin.llf), 1)
    gx = np.linspace(np.log2(3), np.log2(d.tnt_early_ngL.quantile(0.98)), 120)
    age_med = d.age_y.median()
    diff = design(gx, age_med) - design(np.array([np.log2(URL_NGL)]), age_med)
    lp = diff @ spl.params
    se = np.sqrt(np.einsum('ij,jk,ik->i', diff, spl.cov_params(), diff))
    curve = pd.DataFrame({'tnt_ngL': 2 ** gx, 'OR': np.exp(lp), 'lo': np.exp(lp - 1.96 * se), 'hi': np.exp(lp + 1.96 * se)})
    return curve, p_nl, 2 ** knots

curve_poor, p_nonlin, knots_poor = spline_fit(P_TNT, 'poor')
curve_det, p_nonlin_det, knots_det = spline_fit(Dt, 'deterioration')
print(f'\nSpline knots (ng/L), deterioration model: {np.round(knots_det, 1)}; P for non-linearity: {p_nonlin_det:.3g}')
print(f'Spline knots (ng/L), death or critical AMA model: {np.round(knots_poor, 1)}; P for non-linearity: {p_nonlin:.3g}')

In [ ]:
# Figure 3: hs-cTn and outcomes (dose-response).
fig, axs = plt.subplots(1, 3, figsize=(17, 4.9))
fig.subplots_adjust(wspace=0.36)
for ax, t, ylab in [(axs[0], dose, 'Death or critical discharge (%)'), (axs[1], dose_det, 'Deterioration within 7 days (%)')]:
    ax.bar(range(len(t)), t['rate %'], color=BLUE, edgecolor='none', width=0.65)
    ymax = max(t['rate %'].max() * 1.3, 10)
    for i, (n_ev, nn, r) in enumerate(zip(t.events, t.N, t['rate %'])):
        ax.text(i, r + 0.015 * ymax, f'{int(n_ev)}/{int(nn)}', ha='center', va='bottom', fontsize=11)
    ax.set_xticks(range(len(t))); ax.set_xticklabels(labels_c)
    ax.set_xlabel('Early hs-cTnT (ng/L)'); ax.set_ylabel(ylab)
    ax.set_ylim(0, ymax)
ax = axs[2]
band = ax.fill_between(curve_det.tnt_ngL, curve_det.lo, curve_det.hi, color=BLUE_BAND, linewidth=0)
line, = ax.plot(curve_det.tnt_ngL, curve_det.OR, color=BLUE, linewidth=2.2)
h1 = ax.axhline(1, color=REF_INK, linestyle=':', linewidth=1.1)
v14 = Line2D([], [], color=REF_INK, linestyle='--', linewidth=1.1)   # legend entry; the line is drawn after the legend is placed
ax.set_xscale('log'); ax.set_yscale('log')
plain_log_ticks(ax.xaxis, [3, 10, 30, 100, 300, 1000], curve_det.tnt_ngL.min(), curve_det.tnt_ngL.max())
ylo, yhi = ax.get_ylim()
plain_log_ticks(ax.yaxis, [0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32], ylo, yhi)
ax.set_xlabel('Early hs-cTnT (ng/L)'); ax.set_ylabel('Odds ratio, deterioration')
leg = ax.legend([line, band, h1, v14], [f'Age-adjusted odds ratio (reference {URL_NGL} ng/L)', '95% confidence interval', 'Odds ratio = 1',
                                        f'Upper reference limit ({URL_NGL} ng/L)'], loc='upper left', fontsize=9.5)
# Dashed reference line at the URL: stop it just below the legend so the two do not overlap.
xlim_c, ylim_c = ax.get_xlim(), ax.get_ylim()
fig.canvas.draw()
leg_box = leg.get_window_extent().transformed(ax.transAxes.inverted())
x_url = ax.transAxes.inverted().transform(ax.transData.transform((URL_NGL, 1)))[0]
top_url = leg_box.y0 - 0.03 if leg_box.x0 <= x_url <= leg_box.x1 else 1.0
ax.plot([URL_NGL, URL_NGL], [0, top_url], transform=ax.get_xaxis_transform(), color=REF_INK, linestyle='--', linewidth=1.1)
ax.set_xlim(xlim_c); ax.set_ylim(ylim_c)
panel_letters(fig, axs, 'ABC')
save(fig, 'Figure3_hsTn_and_outcomes')
plt.show()

In [ ]:
# Incremental prognostic value of hs-cTn (apparent and optimism-corrected AUC; 1,000 bootstrap samples).
def fit_auc(df, f, outcome):
    m = smf.glm(f, data=df, family=sm.families.Binomial()).fit()
    return m, roc_auc_score(df[outcome], m.predict(df))

def incremental(df, base, outcome, reps=1000):
    m0, a0 = fit_auc(df, base, outcome)
    m1, a1 = fit_auc(df, base + ' + log2_tnt', outcome)
    lr = stats.chi2.sf(2 * (m1.llf - m0.llf), 1)
    deltas, o0, o1 = [], [], []
    for _ in range(reps):
        s_ = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9)))
        if s_[outcome].nunique() < 2:
            continue
        try:
            b0, x0 = fit_auc(s_, base, outcome); b1, x1 = fit_auc(s_, base + ' + log2_tnt', outcome)
            deltas.append(x1 - x0)
            o0.append(x0 - roc_auc_score(df[outcome], b0.predict(df)))
            o1.append(x1 - roc_auc_score(df[outcome], b1.predict(df)))
        except Exception:
            pass
    return {'AUC base': a0, 'AUC base (optimism-corrected)': a0 - np.mean(o0), 'AUC + hs-cTn': a1,
            'AUC + hs-cTn (optimism-corrected)': a1 - np.mean(o1), 'delta AUC': a1 - a0,
            'delta lo': np.percentile(deltas, 2.5), 'delta hi': np.percentile(deltas, 97.5), 'LR test P': lr, 'reps': len(deltas)}

inc_poor = incremental(prep_det(P_TNT), 'poor ~ age_y + male + group_ane + group_sae + ventilation + vasoactive + status_epilepticus', 'poor')
inc_det = incremental(Dt, 'deterioration ~ age_y + male + group_ane + group_sae + status_epilepticus', 'deterioration')
table4 = pd.DataFrame([
    dict(outcome='Death or critical AMA (primary cohort)', base='age, sex, syndrome group, ventilation, vasoactive infusion, status epilepticus', **inc_poor),
    dict(outcome='Deterioration within 7 days (no organ support at testing)', base='age, sex, syndrome group, status epilepticus', **inc_det),
])
num_cols = table4.select_dtypes('number').columns.drop(['reps'])
table4[num_cols] = table4[num_cols].round(4)
table4

In [ ]:
# Figure 2: forest plot of adjusted relative risks.
def num(row):
    return tuple(row['_num'])
inj = assoc.set_index('exposure')
sections = [
    ('Death or critical discharge', [
        ('Myocardial injury, model 1', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 1_num']),
        ('Myocardial injury, model 2', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 2_num']),
        ('Myocardial injury, model 3', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 3_num']),
        ('Myocardial injury, support before testing', tuple(rr_pre[:3])),
        ('Myocardial injury, overlap weighting', (ow['RR'], ow['RR_lo'], ow['RR_hi'])),
        ('LVEF < 55%, model 3', inj.loc['LVEF < 55%', 'Model 3_num']),
        ('NT-proBNP per doubling, model 3', inj.loc['NT-proBNP, per doubling', 'Model 3_num']),
        ('QTcF ≥ 460 ms, model 3', inj.loc['QTc (Fridericia) >= 460 ms', 'Model 3_num']),
    ]),
    ('Deterioration within 7 days, no organ support at testing', [
        ('Myocardial injury, adjusted', num(table5.iloc[1])),
        ('Myocardial injury, adjusted incl. lactate', num(table5.iloc[2])),
        ('hs-cTnT per doubling, adjusted', num(table5.iloc[3])),
    ]),
]
rows_fp = []
for head, items in sections:
    rows_fp.append((head, None))
    rows_fp += [(lab, v) for lab, v in items]
fig, ax = plt.subplots(figsize=(10, 0.5 * len(rows_fp) + 1.0))
ys = np.arange(len(rows_fp))[::-1]
for y, (lab, v) in zip(ys, rows_fp):
    if v is None:
        continue
    rr, lo, hi = v
    ax.plot([lo, hi], [y, y], color=BLUE, linewidth=1.6)
    ax.plot(rr, y, marker='s', color=BLUE, markersize=8)
    ax.annotate(f'{rnd(rr, 2)} ({rnd(lo, 2)}–{rnd(hi, 2)})', xy=(1.02, y), xycoords=('axes fraction', 'data'), va='center', fontsize=11)
ax.set_yticks(ys)
ax.set_yticklabels([lab for lab, _ in rows_fp])
for tick, (lab, v) in zip(ax.get_yticklabels(), rows_fp):
    if v is None:
        tick.set_fontweight('bold')
ax.set_xscale('log')
ref = ax.axvline(1, color=REF_INK, linestyle='--', linewidth=1.1)
xlo, xhi = ax.get_xlim()
plain_log_ticks(ax.xaxis, [0.25, 0.5, 1, 2, 4, 8], xlo, xhi)
ax.set_xlabel('Relative risk (95% CI)')
ax.legend([ref], ['Relative risk = 1'], loc='lower right')
save(fig, 'Figure2_forest_adjusted_relative_risks')
plt.show()

In [ ]:
# Exploratory brain-heart correlates: neuroimaging and EEG features from the ITES evidence (index admission).
BH = P_TNT.copy()
BH['thalamus'] = np.where(BH.mri_n.notna() | BH.ct_n.notna(), (BH.mri_thal_bilat.fillna(False).astype(bool) | BH.ct_thal_bilat.fillna(False).astype(bool)), np.nan)
BH['brainstem'] = np.where(BH.mri_n.notna(), BH.mri_brainstem.fillna(False).astype(bool), np.nan)
BH['diffuse'] = np.where(BH.mri_n.notna() | BH.ct_n.notna(), (BH.mri_diffuse_edema.fillna(False).astype(bool) | BH.ct_diffuse_edema.fillna(False).astype(bool)), np.nan)
BH['eeg_suppressed'] = np.where(BH.eeg_n.notna(), BH.eeg_suppressed.fillna(False).astype(bool), np.nan)
bh_rows = []
for lab, col in [('Bilateral thalamic lesions', 'thalamus'), ('Brainstem lesion', 'brainstem'), ('Diffuse cerebral edema', 'diffuse'),
                 ('EEG background suppression', 'eeg_suppressed')]:
    d = BH[BH[col].notna()].copy(); d[col] = d[col].astype(int); d['inj'] = d.injury.astype(int)
    a1, n1 = int(d[d[col] == 1].inj.sum()), int((d[col] == 1).sum())
    a0, n0 = int(d[d[col] == 0].inj.sum()), int((d[col] == 0).sum())
    try:
        m = smf.glm(f'inj ~ {col} + age_y + vasoactive', data=d, family=sm.families.Poisson()).fit(cov_type='HC0')
        rr, lo, hi, p = rr_row(m, col)
        est = f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})'
    except Exception:
        est, p = 'not estimable', np.nan
    bh_rows.append({'feature': lab, 'N with study': len(d), 'injury if present': f'{a1}/{n1}', 'injury if absent': f'{a0}/{n0}',
                    'adjusted RR (age, vasoactive)': est, 'P': p, '_p1': a1 / n1 if n1 else np.nan, '_p0': a0 / n0 if n0 else np.nan,
                    '_n': (a1, n1, a0, n0)})
brain_heart = pd.DataFrame(bh_rows)
brain_heart.drop(columns=['_p1', '_p0', '_n'])

In [ ]:
# Supplementary Figure S1 (exploratory): prevalence of myocardial injury by neuroimaging and EEG findings.
fig, ax = plt.subplots(figsize=(10, 5))
xpos = np.arange(len(brain_heart))
w_ = 0.36
b1 = ax.bar(xpos - w_ / 2, 100 * brain_heart._p1, width=w_, color=BLUE, edgecolor='white', linewidth=1.5)
b0 = ax.bar(xpos + w_ / 2, 100 * brain_heart._p0, width=w_, color=ORANGE, edgecolor='white', linewidth=1.5)
for i, (a1, n1, a0, n0) in enumerate(brain_heart._n):
    ax.text(i - w_ / 2, 100 * a1 / n1 + 1.2 if n1 else 1, f'{a1}/{n1}', ha='center', va='bottom', fontsize=10)
    ax.text(i + w_ / 2, 100 * a0 / n0 + 1.2 if n0 else 1, f'{a0}/{n0}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(xpos); ax.set_xticklabels(['Bilateral thalamic\nlesions', 'Brainstem\nlesion', 'Diffuse cerebral\nedema', 'EEG\nsuppression'])
ax.set_ylabel(f'hs-cTnT > {URL_NGL} ng/L (%)')
ax.set_ylim(0, np.nanmax(100 * np.r_[brain_heart._p1, brain_heart._p0]) * 1.3)
ax.legend([b1, b0], ['Finding present', 'Finding absent'], loc='upper right')
save(fig, 'FigureS1_brain_heart_correlates')
plt.show()

In [ ]:
# Sensitivity analyses for myocardial injury (Model 3 unless stated).
def model3_rr(d, var='injury', outcome='poor', rhs=M3):
    d = d.copy(); d[var] = d[var].astype(int); d[outcome] = d[outcome].astype(int)
    m = smf.glm(f'{outcome} ~ {var}{rhs}', data=d, family=sm.families.Poisson()).fit(cov_type='HC0')
    return rr_row(m, var)

def prep(d):
    d = d.copy()
    for c in ['male', 'age_lt1', 'ventilation', 'vasoactive', 'status_epilepticus']:
        d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int); d['group_sae'] = (d.group3 == 'SAE').astype(int)
    return d

sens = []
def add(lab, d, **kw):
    try:
        rr, lo, hi, p = model3_rr(d, **kw)
        sens.append((lab, len(d), int(d[kw.get('outcome', 'poor')].sum()), f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})', p))
    except Exception as e:
        sens.append((lab, len(d), int(d[kw.get('outcome', 'poor')].sum()), f'not estimable ({type(e).__name__})', np.nan))

add('Primary (early hs-cTn, ICU, arrest rule)', P_TNT)
pk = prep(icu[icu.tnt_peak.notna() & ~icu.arrest_before_tnt_first])
pk['injury'] = pk.tnt_peak > HSCTN_URL
add('Peak hs-cTn during admission', pk)
add('Excluding every patient with any cardiac arrest/CPR', P_TNT[~P_TNT.cpr_any])
add('Age >= 1 year only', P_TNT[P_TNT.age_y >= 1])
d = P_TNT.copy(); d['death_any_ama'] = (d.death | d.ama).astype(int)
add('Outcome: death or any discharge against advice', d, outcome='death_any_ama')
add(f'Injury defined as hs-cTn > {ALT_NGL} ng/L', P_TNT.assign(injury=P_TNT.injury_alt))
d = P_TNT.copy(); d['poor_rule'] = (d.death.astype(bool) | d.critical_ama_rule.astype(bool)).astype(int)
add('Outcome: critical discharge by the documentation rule (no clinician review)', d, outcome='poor_rule')
add('Consensus ITES only (definite, related or possible)', P_TNT[P_TNT.ites_any.astype(bool)], rhs=M3.replace(' + group_sae', ''))
dd = P_TNT.copy(); dd['death_i'] = dd.death.astype(int)
e1, n1 = int(dd[dd.injury].death_i.sum()), int(dd.injury.sum()); e0, n0 = int(dd[~dd.injury].death_i.sum()), int((~dd.injury).sum())
sens.append(('Outcome: in-hospital death only (Fisher exact, unadjusted)', len(dd), int(dd.death_i.sum()),
             f'{e1}/{n1} vs {e0}/{n0}', stats.fisher_exact([[e1, n1 - e1], [e0, n0 - e0]])[1]))
lg = smf.glm('poor ~ injury' + M3, data=P_TNT.assign(injury=P_TNT.injury.astype(int)), family=sm.families.Binomial()).fit()
sens.append(('Logistic regression (odds ratio), Model 3', len(P_TNT), int(P_TNT.poor.sum()),
             f'{rnd(np.exp(lg.params["injury"]), 2)} ({rnd(np.exp(lg.conf_int().loc["injury", 0]), 2)}-{rnd(np.exp(lg.conf_int().loc["injury", 1]), 2)})', lg.pvalues['injury']))
sensitivity = pd.DataFrame(sens, columns=['analysis', 'N', 'events', 'estimate (95% CI)', 'P'])
sensitivity

In [ ]:
# Supplementary: CK-MB activity versus hs-cTn within the primary cohort.
C = P_TNT[P_TNT.ckmb_early.notna()].copy()
C['ckmb_high'] = C.ckmb_early > 25
disc = pd.crosstab(C.ckmb_high.map({True: 'CK-MB activity > 25 U/L', False: 'CK-MB activity <= 25 U/L'}),
                   C.injury.map({True: f'hs-cTn > {URL_NGL} ng/L', False: f'hs-cTn <= {URL_NGL} ng/L'}))
auc_ckmb = roc_auc_score(C.poor, C.ckmb_early); auc_tnt_c = roc_auc_score(C.poor, C.tnt_early)
print(f'N={len(C)}; AUC for death or critical AMA: CK-MB activity {auc_ckmb:.3f} vs hs-cTn {auc_tnt_c:.3f}')
disc

In [ ]:
# Numbers quoted in the manuscript (computed here so that every number in the text can be traced and re-checked).
def frac_txt(a, b):
    return f'{int(a)}/{int(b)} ({rnd(100 * a / b)}%)'

def miqr(x, d=1):
    x = pd.Series(x).dropna()
    return f'{rnd(x.median(), d)} ({rnd(x.quantile(.25), d)}-{rnd(x.quantile(.75), d)})'

hrs_first = (P_TNT.tnt_early_time - P_TNT.admit).dt.total_seconds() / 3600
esc7 = P_DET.hours_to_escalation[(P_DET.hours_to_escalation > 0) & (P_DET.hours_to_escalation <= WINDOW_H)]
ites_icu = idx[idx.icu].ites_class.value_counts().to_dict()
MS = {
    'cohort': {'aes_patients': len(coh), 'acute_index': int(coh.acute_index.sum()), 'analysis_base': len(idx),
               'base_age': miqr(idx.age_y), 'base_male': frac_txt(idx.male.sum(), len(idx)), 'base_icu': frac_txt(idx.icu.sum(), len(idx)),
               'base_deaths': int(idx.death.sum()), 'base_critical_ama': int(idx.critical_ama.sum()), 'base_poor': int(idx.poor.sum()),
               'admission_years': [int(idx.admit.dt.year.min()), int(idx.admit.dt.year.max())],
               'icu_tested_48h': frac_txt(icu.tnt_early.notna().sum(), len(icu)),
               'excluded_arrest': int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum()),
               'primary': len(P_TNT), 'primary_age': miqr(P_TNT.age_y), 'primary_age_lt1': frac_txt(P_TNT.age_lt1.sum(), len(P_TNT)),
               'primary_male': frac_txt(P_TNT.male.sum(), len(P_TNT)), 'primary_poor': frac_txt(P_TNT.poor.sum(), len(P_TNT)),
               'primary_deaths': int(P_TNT.death.sum()), 'primary_critical_ama': int(P_TNT.critical_ama.sum()),
               'primary_ventilation': frac_txt(P_TNT.ventilation.sum(), len(P_TNT)), 'primary_vasoactive': frac_txt(P_TNT.vasoactive.sum(), len(P_TNT)),
               'hours_admission_to_first_hsctn': miqr(hrs_first)},
    'ites_all': ev.ites_class.value_counts().to_dict(), 'ites_tier_all': ev.ites_tier.value_counts().to_dict(),
    'ites_icu': ites_icu, 'ites_primary': P_TNT.ites_class.value_counts().to_dict(), 'groups_primary': P_TNT.group.value_counts().to_dict(),
    'ites_review_flags': int(ev.needs_review.sum()),
    'injury': {'primary': frac_txt(P_TNT.injury.sum(), len(P_TNT)), 'alt_threshold': frac_txt(P_TNT.injury_alt.sum(), len(P_TNT)),
               'hsctn_median': miqr(P_TNT.tnt_early_ngL), 'by_group': {g: list(v) for g, v in pheno.set_index('group')['_inj'].items()},
               'lvef_lt55': frac_txt(P_EF.dysfunction.sum(), len(P_EF)), 'lvef_by_group': {g: list(v) for g, v in pheno.set_index('group')['_dys'].items()},
               'qtcf_ge460': frac_txt(P_QT.qt_prolonged.sum(), len(P_QT)), 'qt_by_group': {g: list(v) for g, v in pheno.set_index('group')['_qt'].items()},
               'bnp_median': miqr(P_BNP.bnp_early, 0), 'p_groups': {'injury': p_inj, 'hsctn': p_tnt, 'lvef': p_dys, 'qtc': p_qt, 'bnp': p_bnp}},
    'death_or_critical_ama': {'injured': frac_txt(P_TNT[P_TNT.injury].poor.sum(), P_TNT.injury.sum()),
                              'not_injured': frac_txt(P_TNT[~P_TNT.injury].poor.sum(), (~P_TNT.injury).sum()),
                              'models': assoc[['exposure', 'Model 1', 'Model 1 P', 'Model 2', 'Model 2 P', 'Model 3', 'Model 3 P']].to_dict('records'),
                              'overlap_weighting': {k: (float(v) if not isinstance(v, int) else v) for k, v in ow.items()},
                              'adjusted_support_before_testing': [float(v) for v in rr_pre],
                              'dose': dose.reset_index().astype({'cat': str}).to_dict('records'), 'p_trend': p_trend},
    'deterioration': {'n': len(P_DET), 'events': int(P_DET.deterioration.sum()), 'vaso_7d': int(P_DET.vaso_7d.sum()),
                      'vent_7d': int(P_DET.vent_7d.sum()), 'death_or_critical_ama': int(P_DET.poor.sum()),
                      'injured': frac_txt(P_DET[P_DET.injury].deterioration.sum(), P_DET.injury.sum()),
                      'not_injured': frac_txt(P_DET[~P_DET.injury].deterioration.sum(), (~P_DET.injury).sum()),
                      'hours_to_escalation_within_7d': miqr(esc7), 'n_escalations_within_7d': int(len(esc7)),
                      'table5': table5.drop(columns=['_num']).to_dict('records'), 'p_interaction_ites': p_interaction,
                      'subgroups': det_subgroups.to_dict('records'), 'dose': dose_det.reset_index().astype({'cat': str}).to_dict('records'),
                      'p_trend': p_trend_det, 'p_nonlinearity': p_nonlin_det, 'incremental': inc_det},
    'incremental_poor': inc_poor,
    'brain_heart': brain_heart.drop(columns=['_p1', '_p0', '_n']).to_dict('records'),
    'ckmb': {'n': len(C), 'auc_ckmb': auc_ckmb, 'auc_hsctn': auc_tnt_c, 'crosstab': disc.to_dict()},
    'sensitivity_poor': sensitivity.to_dict('records'),
    'measured_n_primary': {c: int(P_TNT[c].notna().sum()) for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'measured_n_primary_by_injury': {c: [int(P_TNT[P_TNT.injury][c].notna().sum()), int(P_TNT[~P_TNT.injury][c].notna().sum())]
                                     for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'measured_n_det': {c: int(P_DET[c].notna().sum()) for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'admission_range': [idx.admit.min().strftime('%Y-%m'), idx.admit.max().strftime('%Y-%m')],   # months only: no exact admission dates in outputs
    'qtcb_ge460': frac_txt(P_QT.qtb_prolonged.sum(), len(P_QT)), 'qtb_by_group': {g: list(v) for g, v in pheno.set_index('group')['_qtb'].items()},
    'p_qtcb_groups': p_qtb, 'p_groups_monte_carlo': p_mc,
    'deterioration_poor_7d': int(P_DET.poor_7d.sum()), 'deterioration_poor_any': int(P_DET.poor.astype(bool).sum()),
    'asem_candidates': ev[(ev.mri_diffuse_edema.fillna(False).astype(bool) | ev.ct_diffuse_edema.fillna(False).astype(bool)) & ev.shock & (ev.mof_count >= 3)]
                       .ites_class.value_counts().to_dict(),
    'definite_ites_without_fever': int((ev.ites_tier.eq('Definite ITES') & ~ev.fever).sum()),
    'definite_ites_without_altered_consciousness': int((ev.ites_tier.eq('Definite ITES') & ~ev.encephalopathy_text).sum()),
    'arrest_excluded_tnt': int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum()),
    'fragility_index_deterioration': int(frag_index), 'fisher_p_deterioration': float(frag_fisher_p),
    'dama_review_index': idx[idx.ama].vis.map(FZ.dama_review_decision).fillna('not reviewed').value_counts().to_dict(),
    'critical_ama_final_vs_rule': [int(idx.critical_ama.sum()), int(idx.critical_ama_rule.sum())],
    'kruskal_note': 'hs-cTn across groups: Kruskal-Wallis; proportions: chi-square',
}
(OUT / 'manuscript_numbers.json').write_text(json.dumps(MS, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Hours from admission to first hs-cTn:', MS['cohort']['hours_admission_to_first_hsctn'], '| escalation within 7 days, hours:', MS['deterioration']['hours_to_escalation_within_7d'])

In [ ]:
# Export tables, cohort file, and key numbers.
with pd.ExcelWriter(OUT / 'Tables_AES_cardiac.xlsx') as xw:
    flow.to_excel(xw, sheet_name='Cohort_flow', index=False)
    table1.to_excel(xw, sheet_name='Table1_by_injury', index=False)
    table2.to_excel(xw, sheet_name='Table2_phenotype', index=False)
    table3.to_excel(xw, sheet_name='Table3_associations', index=False)
    table4.to_excel(xw, sheet_name='Table4_incremental_AUC', index=False)
    table5.drop(columns=['_num']).to_excel(xw, sheet_name='Table5_time_anchored', index=False)
    pd.DataFrame([ow]).to_excel(xw, sheet_name='Overlap_weighting', index=False)
    balance.to_excel(xw, sheet_name='S1_PS_balance', index=False)
    tab_tested.to_excel(xw, sheet_name='S2_tested_vs_not', index=False)
    sensitivity.to_excel(xw, sheet_name='S3_sensitivity', index=False)
    brain_heart.drop(columns=['_p1', '_p0', '_n']).to_excel(xw, sheet_name='S4_brain_heart', index=False)
    disc.to_excel(xw, sheet_name='S5_CKMB_vs_hsTn')
    dose.reset_index().to_excel(xw, sheet_name='S6_dose_death_critAMA', index=False)
    dose_det.reset_index().to_excel(xw, sheet_name='S7_dose_deterioration', index=False)
    table_det.to_excel(xw, sheet_name='S8_deterioration_cohort', index=False)
    det_subgroups.to_excel(xw, sheet_name='S9_deterioration_by_ITES', index=False)
keep = ['patient', 'vis', 'subtype', 'ites_class', 'ites_tier', 'group', 'admit', 'year', 'age_y', 'male', 'icu', 'death', 'ama', 'critical_ama', 'poor',
        'tnt_early_ngL', 'tnt_peak_ngL', 'bnp_early', 'lac_early', 'qtcf_early', 'qtcb_peak', 'lvef_early_min', 'lvef_min',
        'ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock', 'cpr_any', 'methylprednisolone', 'ivig', 'tocilizumab']
out_c = idx[keep].copy()
out_c['in_primary_cohort'] = out_c.vis.isin(set(P_TNT.vis))
out_c['myocardial_injury'] = out_c.vis.map(P_TNT.set_index('vis').injury)
out_c['in_deterioration_cohort'] = out_c.vis.isin(set(P_DET.vis))
out_c['deterioration_7d'] = out_c.vis.map(P_DET.set_index('vis').deterioration)
out_c.to_csv(OUT / 'aes_cardiac_cohort.csv', index=False, encoding='utf-8-sig')
key = {'hsctn_url_ngL': URL_NGL, 'hsctn_alt_ngL': ALT_NGL, 'index_admissions': len(idx), 'icu': len(icu), 'primary_cohort': len(P_TNT), 'primary_events': int(P_TNT.poor.sum()),
       'injury_n': int(P_TNT.injury.sum()), 'deterioration_cohort': len(P_DET), 'deterioration_events': int(P_DET.deterioration.sum()),
       'p_trend_poor': p_trend, 'p_trend_deterioration': p_trend_det, 'p_nonlinearity_poor': p_nonlin, 'p_nonlinearity_det': p_nonlin_det,
       'overlap_weighting': ow, 'rr_adjusted_for_support_before_testing': list(rr_pre),
       'incremental_poor': inc_poor, 'incremental_deterioration': inc_det,
       'hours_to_escalation_median_any_time': float(P_DET.hours_to_escalation.median()),
       'hours_to_escalation_iqr_any_time': [float(v) for v in P_DET.hours_to_escalation.quantile([.25, .75])],
       'hours_to_escalation_within_7d': MS['deterioration']['hours_to_escalation_within_7d'],
       'p_interaction_injury_x_ites': p_interaction,
       'p_across_groups': {'injury': p_inj, 'tnt': p_tnt, 'lvef': p_dys, 'qtc': p_qt}}
(OUT / 'key_numbers.json').write_text(json.dumps(key, indent=2, default=float), encoding='utf-8')
print('Saved to', OUT)